# КосмоХакатон 2026 — Мониторинг природных пожаров
## Единый пайплайн: обучение + инференс + сабмишн

**Как пользоваться: Runtime → Run all. Больше ничего менять не нужно.**

Ноутбук делает всё подряд и на выходе кладёт `submission.csv` в `CHECKPOINT_DIR`
(на Colab это Google Drive, файл переживёт закрытие рантайма):

| # | Этап | Прибл. время |
|---|---|---|
| 1–3 | установка, загрузка данных с Kaggle, встроенный код | ~5 мин |
| 4 | Модуль AF: активное горение (LightGBM-ансамбль) | ~8 мин |
| 5 | Модуль BS: попиксельный LightGBM | ~6 мин |
| 6 | Модуль BS: OOF-карты для стекинга | ~12 мин |
| 7 | Модуль BS: пространственный уточнитель на GPU | ~15 мин |
| 8 | Калибровка стратегии и порогов на валидации | ~3 мин |
| 9 | **Инференс на тесте (замеряется отдельно)** | ~2–4 мин |
| 10 | Отчёт по времени + упаковка весов | <1 мин |

Итого ориентировочно **50–60 минут**. Нужен GPU-рантайм (T4 хватает, A100 быстрее).

### Что считается «скоростью» в критериях
Организаторы замеряют **инференс**, а не обучение. Поэтому секция 9 —
самодостаточный замеряемый блок: он заново поднимает модели с диска и
прогоняет весь тест, ровно как это сделает `inference.py` у них.

## 1. Установка и окружение

In [ ]:
import sys, subprocess

# lightgbm есть не во всех образах Colab; torch/torchvision есть всегда.
for _pkg, _imp in [('lightgbm', 'lightgbm'), ('kagglehub', 'kagglehub'),
                    ('scikit-image', 'skimage'), ('rasterio', 'rasterio')]:
    try:
        __import__(_imp)
    except ImportError:
        print('ставим', _pkg, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
print('окружение готово')

In [ ]:
import json, os, time, warnings
from contextlib import contextmanager

import numpy as np
import pandas as pd
import torch
import lightgbm as lgb
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('ВНИМАНИЕ: GPU не найден. Пространственный уточнитель будет обучаться '
          'очень долго. Runtime -> Change runtime type -> GPU.')

### Секундомер

Все тяжёлые секции обёрнуты в `stage(...)`, чтобы в конце (секция 10) выдать
честную таблицу времени и отдельно — время инференса, по которому начисляются
баллы за скорость.

In [ ]:
TIMINGS = {}


@contextmanager
def stage(name):
    print(f'>>> {name} ...', flush=True)
    t0 = time.time()
    yield
    dt = time.time() - t0
    TIMINGS[name] = dt
    print(f'<<< {name}: {dt:.1f} с', flush=True)

## 2. Данные с Kaggle

Датасет публичный: https://www.kaggle.com/datasets/shotme/cosmohack2026

**Нужны Kaggle-креды** даже для публичного датасета: Colab → значок ключа
слева (Secrets) → Add new secret → `KAGGLE_USERNAME` и `KAGGLE_KEY`
(kaggle.com → Settings → API → Create New Token) → включить Notebook access.
Виджет `kagglehub.login()` здесь намеренно не вызывается: он асинхронный и
ломает Run All.

In [ ]:
KAGGLE_DATASET_HANDLE = 'shotme/cosmohack2026'
_LOCAL_DEV_PATH = r'..\..\Мониторинг DATA\Мониторинг DATA'  # только для локальной отладки

with stage('1. загрузка данных'):
    if os.path.isdir(_LOCAL_DEV_PATH):
        DATASET_DIR = _LOCAL_DEV_PATH
        print('локальные данные:', DATASET_DIR)
    else:
        import kagglehub
        try:
            DATASET_DIR = kagglehub.dataset_download(KAGGLE_DATASET_HANDLE)
        except Exception as e:
            raise RuntimeError(
                'Не удалось скачать датасет с Kaggle - почти наверняка не настроены '
                'credentials. Colab -> Secrets -> KAGGLE_USERNAME и KAGGLE_KEY '
                '(kaggle.com -> Settings -> API -> Create New Token) -> включить '
                'Notebook access для обоих.'
            ) from e
        print('датасет:', DATASET_DIR)

In [ ]:
def _setup_checkpoint_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        path = '/content/drive/MyDrive/kosmohack2026/checkpoints'
    except ImportError:
        path = os.path.join('.', 'checkpoints')
    os.makedirs(path, exist_ok=True)
    return path


CHECKPOINT_DIR = _setup_checkpoint_dir()
print('чекпоинты:', CHECKPOINT_DIR)

## 3. Код решения (встроен из `src/`)

Ноутбук самодостаточен: рядом с ним на Colab нет репозитория, поэтому модули
вшиты целиком, а не импортируются. Эти же файлы лежат в `solution/src/` и
используются standalone-скриптом `inference.py`, который будут запускать
организаторы.

In [ ]:
# === встроено из src/dataio.py - не редактировать здесь, правьте src/dataio.py и пересоберите через build_pipeline_notebook.py ===
"""Chip readers for the КосмоХакатон fire-monitoring dataset.

Disk on the dev machine is essentially full, so `TarChipSource` reads every
GeoTIFF straight out of `fire-train-renamed.tar` into memory (via
rasterio.io.MemoryFile) and never extracts anything to disk. `DirChipSource`
is the same interface for a plain extracted folder, for when the organizers'
test set turns out to ship as loose files instead of a tar - swap the source,
keep every notebook/inference cell unchanged.

Band names are read from each GeoTIFF's own band descriptions (they're
embedded in the files), not hardcoded from the case statement PDF, so this
keeps working even if a channel gets reordered.
"""
from __future__ import annotations

import io
import tarfile
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import rasterio
from rasterio.io import MemoryFile


class _ChipSourceBase:
    def _read_bytes(self, relpath: str) -> bytes:
        raise NotImplementedError

    def close(self) -> None:
        """No-op by default; TarChipSource overrides this to close its tarfile handle."""

    def __enter__(self):
        return self

    def __exit__(self, *exc) -> None:
        self.close()

    def read_named(self, relpath: str) -> dict:
        """Multi-band GeoTIFF -> dict keyed by band description, plus '_bands' (C,H,W),
        '_names' (tuple) and '_profile' (rasterio profile, has crs/transform/nodata)."""
        raw = self._read_bytes(relpath)
        with MemoryFile(raw) as mem, mem.open() as src:
            names = src.descriptions
            arr = src.read()
            profile = src.profile
        out = {"_bands": arr, "_names": tuple(names), "_profile": profile}
        for i, name in enumerate(names):
            if name:
                out[name] = arr[i]
        return out

    def read_csv(self, relpath: str) -> pd.DataFrame:
        return pd.read_csv(io.BytesIO(self._read_bytes(relpath)))

    def meta(self, kind: str) -> pd.DataFrame:
        return self.read_csv(f"{kind}/meta.csv")

    def chip_ids(self, kind: str) -> list[str]:
        return self.meta(kind)["chip_id"].tolist()

    def _try_load_mask(self, relpath: str) -> tuple[np.ndarray | None, float | None]:
        """Test chips ship with no mask (that's what we predict) - returns (None, None)
        instead of raising, so load_af/load_bs work unchanged on train, val AND test."""
        try:
            mask_raw = self.read_named(relpath)
        except FileNotFoundError:
            return None, None
        return mask_raw["_bands"][0], mask_raw["_profile"].get("nodata")

    def load_af(self, chip_id: str) -> dict:
        viirs = self.read_named(f"af/viirs/{chip_id}_VIIRS_I1-I5.tif")
        aux = self.read_named(f"af/aux/{chip_id}_AUX.tif")
        mask, mask_nodata = self._try_load_mask(f"af/masks/{chip_id}_MASK.tif")
        return {
            "chip_id": chip_id,
            "viirs": viirs,
            "aux": aux,
            "mask": mask,
            "mask_nodata": mask_nodata,
        }

    def load_bs(self, chip_id: str) -> dict:
        s2_pre = self.read_named(f"bs/sentinel2_pre/{chip_id}_Sentinel-2_pre.tif")
        s2_post = self.read_named(f"bs/sentinel2_post/{chip_id}_Sentinel-2_post.tif")
        s1_pre = self.read_named(f"bs/sentinel1_pre/{chip_id}_Sentinel-1_pre.tif")
        s1_post = self.read_named(f"bs/sentinel1_post/{chip_id}_Sentinel-1_post.tif")
        aux = self.read_named(f"bs/aux/{chip_id}_AUX.tif")
        mask, mask_nodata = self._try_load_mask(f"bs/masks/{chip_id}_MASK.tif")
        return {
            "chip_id": chip_id,
            "s2_pre": s2_pre,
            "s2_post": s2_post,
            "s1_pre": s1_pre,
            "s1_post": s1_post,
            "aux": aux,
            "mask": mask,
            "mask_nodata": mask_nodata,
        }


class TarChipSource(_ChipSourceBase):
    """Reads chips directly out of fire-train-renamed.tar, no extraction to disk."""

    def __init__(self, tar_path: str | Path, root: str = "train"):
        self.tar_path = str(tar_path)
        self.root = root
        self._tar = tarfile.open(self.tar_path)

    def _read_bytes(self, relpath: str) -> bytes:
        member = f"{self.root}/{relpath}"
        try:
            buf = self._tar.extractfile(member)
        except KeyError:
            # tarfile raises KeyError (not FileNotFoundError) for a missing member -
            # normalize it so callers (e.g. _try_load_mask) can catch one exception
            # type regardless of which ChipSource they're using.
            raise FileNotFoundError(member) from None
        if buf is None:
            raise FileNotFoundError(member)
        return buf.read()

    def close(self) -> None:
        self._tar.close()


class DirChipSource(_ChipSourceBase):
    """Same interface as TarChipSource, for an already-extracted directory
    (e.g. the organizers' --data-dir at inference time)."""

    def __init__(self, root_dir: str | Path):
        self.root_dir = Path(root_dir)

    def _read_bytes(self, relpath: str) -> bytes:
        path = self.root_dir / relpath
        return path.read_bytes()


def nodata_to_ignore(mask: np.ndarray, nodata: float | None) -> np.ndarray:
    """Boolean valid-pixel mask (True = usable) from a mask array + its nodata value."""
    if nodata is None:
        return np.ones_like(mask, dtype=bool)
    return mask != nodata


def iter_chips(source: _ChipSourceBase, kind: str, chip_ids: Iterable[str] | None = None):
    """Yield load_af()/load_bs() dicts for every chip_id (or all chips if None)."""
    loader = source.load_af if kind == "af" else source.load_bs
    ids = list(chip_ids) if chip_ids is not None else source.chip_ids(kind)
    for chip_id in ids:
        yield loader(chip_id)


In [ ]:
# === встроено из src/rle.py - не редактировать здесь, правьте src/rle.py и пересоберите через build_pipeline_notebook.py ===
"""RLE encode/decode matching the КосмоХакатон submission.csv spec.

Pixel numbering is row-major (left-to-right, top-to-bottom), 1-indexed.
Runs are reported in ascending start order and never overlap or touch,
which is exactly what run-length-encoding a boolean mask gives you for free.
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def encode_rle(mask: np.ndarray) -> str:
    """2D boolean/0-1 array -> RLE string ("start len start len ..."), '' if empty."""
    flat = np.asarray(mask, dtype=bool).ravel(order="C")
    if not flat.any():
        return ""
    padded = np.concatenate(([0], flat.astype(np.int8), [0]))
    diffs = np.diff(padded)
    starts_0idx = np.flatnonzero(diffs == 1)
    ends_k = np.flatnonzero(diffs == -1)
    lengths = ends_k - starts_0idx
    starts_1idx = starts_0idx + 1
    return " ".join(f"{s} {l}" for s, l in zip(starts_1idx, lengths))


def decode_rle(rle, shape: tuple[int, int]) -> np.ndarray:
    """RLE string + (H, W) -> boolean mask.

    An empty mask round-trips through CSV as an empty field, which pandas reads back
    as NaN (float) rather than "" - this is also how the organizers' own
    sample_submission.csv represents "no pixels of this class", so NaN/None are treated
    the same as an empty string here rather than as errors.
    """
    mask = np.zeros(shape[0] * shape[1], dtype=bool)
    if rle is None or (isinstance(rle, float) and np.isnan(rle)):
        rle = ""
    rle = str(rle).strip()
    if rle:
        nums = list(map(int, rle.split()))
        for start, length in zip(nums[0::2], nums[1::2]):
            s0 = start - 1
            mask[s0 : s0 + length] = True
    return mask.reshape(shape, order="C")


def submission_row(chip_id: str, class_id: int, mask: np.ndarray) -> dict:
    return {"chip_id": chip_id, "class_id": int(class_id), "rle": encode_rle(mask)}


def validate_submission(df: pd.DataFrame, sample_submission: pd.DataFrame) -> list[str]:
    """Structural checks against the shipped sample_submission.csv. Empty list == OK."""
    problems = []
    required = set(zip(sample_submission["chip_id"], sample_submission["class_id"]))
    got = set(zip(df["chip_id"], df["class_id"]))
    missing = required - got
    extra = got - required
    if missing:
        problems.append(f"missing {len(missing)} (chip_id, class_id) pairs, e.g. {list(missing)[:5]}")
    if extra:
        problems.append(f"unexpected {len(extra)} (chip_id, class_id) pairs, e.g. {list(extra)[:5]}")
    # NaN in "rle" is NOT an error: an empty mask round-trips through to_csv/read_csv as
    # an empty field, which pandas parses as NaN - decode_rle() treats it as "" on purpose,
    # matching how the organizers' own sample_submission.csv represents an empty mask.
    if df.duplicated(subset=["chip_id", "class_id"]).any():
        problems.append("duplicate (chip_id, class_id) rows found")
    return problems


def validate_submission_full(
    df: pd.DataFrame,
    sample_submission: pd.DataFrame,
    chip_shapes: dict[str, tuple[int, int]],
) -> list[str]:
    """Structural checks plus per-chip overlap/out-of-bounds decoding checks."""
    problems = validate_submission(df, sample_submission)
    for chip_id, group in df.groupby("chip_id"):
        shape = chip_shapes.get(chip_id)
        if shape is None:
            continue
        total = np.zeros(shape, dtype=np.uint8)
        for _, row in group.iterrows():
            try:
                m = decode_rle(row["rle"], shape)
            except Exception as e:  # noqa: BLE001 - want the chip_id in the message
                problems.append(f"{chip_id}/{row['class_id']}: RLE decode failed ({e})")
                continue
            total += m.astype(np.uint8)
        if (total > 1).any():
            problems.append(f"{chip_id}: overlapping masks between classes")
    return problems


In [ ]:
# === встроено из src/metrics.py - не редактировать здесь, правьте src/metrics.py и пересоберите через build_pipeline_notebook.py ===
"""Competition score, computed exactly like the organizers compute it:
pixels from every chip are pooled into one confusion matrix per class
(micro-averaging) *before* precision/recall/F1/IoU are derived - not
averaged per-chip and then across chips.

Score = 0.35 * F1_af + 0.35 * IoU_burn + 0.30 * mIoU_sev
"""
from __future__ import annotations

import numpy as np


class ConfusionAccumulator:
    """Pools TP/FP/FN across an arbitrary number of update() calls (= chips)."""

    def __init__(self, classes: tuple[int, ...]):
        self.classes = tuple(classes)
        self.tp = {c: 0 for c in self.classes}
        self.fp = {c: 0 for c in self.classes}
        self.fn = {c: 0 for c in self.classes}

    def update(self, pred: np.ndarray, true: np.ndarray, valid: np.ndarray | None = None) -> None:
        if valid is None:
            valid = np.ones_like(true, dtype=bool)
        for c in self.classes:
            p = (pred == c) & valid
            t = (true == c) & valid
            self.tp[c] += int(np.sum(p & t))
            self.fp[c] += int(np.sum(p & ~t))
            self.fn[c] += int(np.sum(~p & t))

    def iou(self, c: int) -> float:
        tp, fp, fn = self.tp[c], self.fp[c], self.fn[c]
        denom = tp + fp + fn
        return 1.0 if denom == 0 else tp / denom

    def precision_recall_f1(self, c: int) -> tuple[float, float, float]:
        tp, fp, fn = self.tp[c], self.fp[c], self.fn[c]
        if tp + fp == 0 and tp + fn == 0:
            return 1.0, 1.0, 1.0
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
        return precision, recall, f1


def binarize_burn(mask: np.ndarray) -> np.ndarray:
    """BS mask (0/1/2/3) -> binary burned/not-burned mask, for IoU_burn."""
    return (mask >= 1).astype(np.uint8)


def f1_af(acc: ConfusionAccumulator) -> float:
    return acc.precision_recall_f1(1)[2]


def iou_burn(acc_binary: ConfusionAccumulator) -> float:
    """acc_binary must have been fed binarize_burn() masks with classes=(1,)."""
    return acc_binary.iou(1)


def miou_severity(acc_sev: ConfusionAccumulator) -> float:
    """acc_sev must have been fed the raw 0..3 masks with classes=(1, 2, 3)."""
    return sum(acc_sev.iou(c) for c in (1, 2, 3)) / 3


def score(f1_af_value: float, iou_burn_value: float, miou_sev_value: float) -> float:
    return 0.35 * f1_af_value + 0.35 * iou_burn_value + 0.30 * miou_sev_value


In [ ]:
# === встроено из src/af_pipeline.py - не редактировать здесь, правьте src/af_pipeline.py и пересоберите через build_pipeline_notebook.py ===
"""AF module (active-fire detection) inference pipeline: feature engineering,
LightGBM ensemble, secondary CNN, max-blending, thresholds, postprocessing.

This is the SAME logic as embedded in notebooks/01_af_binary_fire_detection.ipynb
(via build_notebooks.py) - kept here too as a plain importable module so
`inference.py` can run outside Colab/Jupyter, without Kaggle/Google Drive, using
just a local directory of weights. If you change feature engineering or model
architecture here, mirror the change in build_notebooks.py (and vice versa) -
the two are not auto-synced.

Only inference-time code lives here (feature extraction, forward pass, blending,
thresholding, postprocessing, weight load/save). Training-only code (building the
hard-negative-mined pixel table, the LightGBM/CNN training loops, threshold
search) stays in the notebook, since training only ever happens on Colab.
"""
from __future__ import annotations

import json
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.ndimage import uniform_filter

AF_FEATURE_NAMES = [
    "I1", "I2", "I3", "I4", "I5", "diff_I4_I5",
    "z_I4_w9", "z_diff_w9",
    "z_I4_w21", "z_diff_w21", "bg_I4", "bg_diff",
    "z_I4_w41", "z_diff_w41", "z_I4_scale_ratio",
    "solar_zenith", "sensor_zenith", "is_night",
    "I1_I2_ratio", "I2_I3_ratio", "mean_refl_day",
    "landcover", "dem", "t2m", "rh2m", "wind_speed",
    "z_I4_x_cropland", "z_diff_x_urban",
]
AF_LANDCOVER_IDX = AF_FEATURE_NAMES.index("landcover")
AF_LANDCOVER_CROPLAND_CODE = 40  # ESA WorldCover: cropland
AF_LANDCOVER_URBAN_CODE = 50  # ESA WorldCover: built-up


def _fill_nan(arr, fallback=280.0):
    if np.isnan(arr).any():
        fill = np.nanmean(arr) if not np.isnan(arr).all() else fallback
        arr = np.where(np.isnan(arr), fill, arr)
    return arr


def _local_mean_std(arr, win=21):
    mean = uniform_filter(arr, size=win)
    sq_mean = uniform_filter(arr ** 2, size=win)
    std = np.sqrt(np.clip(sq_mean - mean ** 2, 0, None)) + 1e-3
    return mean, std


def extract_af_features(chip, win=21):
    """chip = source.load_af(...). Returns (C, H, W) float32 in AF_FEATURE_NAMES order."""
    v, aux = chip["viirs"], chip["aux"]
    I1, I2, I3 = v["I1"], v["I2"], v["I3"]
    I4, I5 = _fill_nan(v["I4"]), _fill_nan(v["I5"])
    diff = I4 - I5

    bg_I4_9, std_I4_9 = _local_mean_std(I4, 9)
    bg_diff_9, std_diff_9 = _local_mean_std(diff, 9)
    z_I4_w9 = (I4 - bg_I4_9) / std_I4_9
    z_diff_w9 = (diff - bg_diff_9) / std_diff_9

    bg_I4, std_I4 = _local_mean_std(I4, win)
    bg_diff, std_diff = _local_mean_std(diff, win)
    z_I4_w21 = (I4 - bg_I4) / std_I4
    z_diff_w21 = (diff - bg_diff) / std_diff

    bg_I4_41, std_I4_41 = _local_mean_std(I4, 41)
    bg_diff_41, std_diff_41 = _local_mean_std(diff, 41)
    z_I4_w41 = (I4 - bg_I4_41) / std_I4_41
    z_diff_w41 = (diff - bg_diff_41) / std_diff_41
    z_I4_scale_ratio = z_I4_w9 / (np.abs(z_I4_w41) + 1e-3)

    solar_zenith = v["solar_zenith"]
    is_night = (solar_zenith > 90).astype(np.float32)
    I1f = np.nan_to_num(I1, nan=0.0)
    I2f = np.nan_to_num(I2, nan=0.0)
    I3f = np.nan_to_num(I3, nan=0.0)
    I1_I2_ratio = I1f / (I2f + 1e-3)
    I2_I3_ratio = I2f / (I3f + 1e-3)
    mean_refl_day = (I1f + I2f + I3f) / 3.0 * (1 - is_night)

    landcover = aux["landcover"]
    is_cropland = (landcover == AF_LANDCOVER_CROPLAND_CODE).astype(np.float32)
    is_urban = (landcover == AF_LANDCOVER_URBAN_CODE).astype(np.float32)

    feats = np.stack([
        I1, I2, I3, I4, I5, diff,
        z_I4_w9, z_diff_w9,
        z_I4_w21, z_diff_w21, bg_I4, bg_diff,
        z_I4_w41, z_diff_w41, z_I4_scale_ratio,
        solar_zenith, v["sensor_zenith"], is_night,
        I1_I2_ratio, I2_I3_ratio, mean_refl_day,
        landcover, aux["dem"], aux["t2m"], aux["rh2m"], aux["wind_speed"],
        z_I4_w21 * is_cropland, z_diff_w21 * is_urban,
    ], axis=0).astype(np.float32)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


def predict_af_proba(chip, model, win=21):
    """Full-image inference (all pixels, no sampling). Works on test chips (no mask).
    A model trained with objective='none' (focal loss) predicts a raw logit -
    converted to a probability via the `_af_focal` flag set on that Booster."""
    feats = extract_af_features(chip, win=win)
    flat = feats.reshape(feats.shape[0], -1).T
    raw = model.predict(flat, num_iteration=model.best_iteration)
    if getattr(model, "_af_focal", False):
        raw = 1.0 / (1.0 + np.exp(-raw))
    return raw.reshape(feats.shape[1:])


def predict_af_proba_ensemble(chip, models, win=21):
    """Equal-weight average across the LightGBM ensemble (xView2 1st place pattern)."""
    probs = [predict_af_proba(chip, m, win=win) for m in models]
    return np.mean(probs, axis=0)


class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class AFCNN(nn.Module):
    """Compact U-Net for binary fire segmentation - a second, spatial opinion on
    top of the per-pixel LightGBM ensemble."""

    def __init__(self, in_channels, base=16):
        super().__init__()
        self.input_norm = nn.BatchNorm2d(in_channels, affine=False)
        self.enc1 = ConvBlock(in_channels, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out_conv = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        x = self.input_norm(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1).squeeze(1)  # (N, H, W) logits


def predict_af_proba_blend(chip, models, cnn_model, win=21, device="cpu"):
    """max() of the LightGBM ensemble and the CNN, not their average - the
    DrivenData STAC Overflow 1st-place pattern: max of two independent models
    catches pixels either one misses on its own. cnn_model=None -> LightGBM only
    (this is what training picks when the CNN turns out weaker on validation -
    see notebooks/01_af_binary_fire_detection.ipynb section 10)."""
    p_lgb = predict_af_proba_ensemble(chip, models, win=win)
    if cnn_model is None:
        return p_lgb
    feats = extract_af_features(chip, win=win)
    with torch.no_grad():
        p_cnn = torch.sigmoid(
            cnn_model(torch.from_numpy(feats).unsqueeze(0).to(device))
        ).squeeze(0).cpu().numpy()
    return np.maximum(p_lgb, p_cnn)


def apply_af_thresholds(proba, landcover, global_threshold, per_landcover_thresholds=None):
    if not per_landcover_thresholds:
        return proba >= global_threshold
    thr_map = np.full(landcover.shape, global_threshold, dtype=np.float32)
    for code, t in per_landcover_thresholds.items():
        thr_map[landcover == int(code)] = t
    return proba >= thr_map


def confidence_tiered_filter(proba, pred_mask, high_conf=0.9, min_blob_size=3):
    """Drop a small connected component only if it's ALSO low-confidence; a
    single high-confidence pixel is kept untouched (median positive AF chip is
    only ~21 pixels - a flat area filter would erase real small fires)."""
    labeled, n = ndi.label(pred_mask)
    keep = np.zeros_like(pred_mask, dtype=bool)
    for i in range(1, n + 1):
        blob = labeled == i
        if blob.sum() >= min_blob_size or proba[blob].max() >= high_conf:
            keep |= blob
    return keep


def predict_af_mask(chip_id, models, cnn_model, global_threshold, per_landcover_thresholds,
                     source, device="cpu"):
    """Full AF pipeline for one chip -> boolean mask. RLE-encoding is the caller's
    job (`submission_row(chip_id, 1, mask)` from src/rle.py), kept out of this
    module so it has no dependency on rle.py's import context."""
    chip = source.load_af(chip_id)
    proba = predict_af_proba_blend(chip, models, cnn_model, device=device)
    pred_mask = apply_af_thresholds(proba, chip["aux"]["landcover"], global_threshold,
                                     per_landcover_thresholds)
    return confidence_tiered_filter(proba, pred_mask)


def save_af_artifacts(weights_dir, models, cnn_model, global_threshold, per_landcover_thresholds,
                       use_cnn_in_blend, best_iterations, focal_model_idx, seed):
    """Mirrors exactly what the training notebook writes to CHECKPOINT_DIR - a
    weights folder saved this way can be copied straight into `weights/af/` for
    `inference.py` (or vice versa: files downloaded from Colab's CHECKPOINT_DIR
    already have the right names, drop them in unchanged)."""
    os.makedirs(weights_dir, exist_ok=True)
    for i, m in enumerate(models):
        m.save_model(os.path.join(weights_dir, f"af_lightgbm_{i}.txt"))
    if cnn_model is not None:
        torch.save(cnn_model.state_dict(), os.path.join(weights_dir, "af_cnn_best.pt"))
    meta = {
        "feature_names": AF_FEATURE_NAMES,
        "categorical_feature_idx": [AF_LANDCOVER_IDX],
        "n_seeds": len(models),
        "best_iterations": best_iterations,
        "focal_model_idx": focal_model_idx,
        "seed": seed,
        "global_threshold": float(global_threshold),
        "per_landcover_thresholds": per_landcover_thresholds,
        "use_cnn_in_blend": use_cnn_in_blend,
    }
    with open(os.path.join(weights_dir, "af_lightgbm_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def _load_booster(lgb, path):
    """Loads a LightGBM model WITHOUT handing the path to the C library.

    `lgb.Booster(model_file=...)` opens the file inside LightGBM's native code,
    which on Windows uses the system codepage and fails outright on any path
    containing non-ASCII characters - a checkout under "C:\\Users\\...\\Рабочий
    стол\\..." raises "Could not open <path>" even though the file is there and
    os.path.isfile() says True. Reading the text in Python and passing
    `model_str` sidesteps the native file handling entirely.
    """
    with open(path, encoding="utf-8") as f:
        return lgb.Booster(model_str=f.read())


def load_af_artifacts(weights_dir, device="cpu"):
    """Returns a dict: models (list of lgb.Booster), cnn_model (AFCNN or None),
    global_threshold (float), per_landcover_thresholds (dict or None)."""
    import lightgbm as lgb  # local import: only needed at inference time

    meta_path = os.path.join(weights_dir, "af_lightgbm_meta.json")
    with open(meta_path, encoding="utf-8") as f:
        meta = json.load(f)

    models = []
    for i in range(meta["n_seeds"]):
        m = _load_booster(lgb, os.path.join(weights_dir, f"af_lightgbm_{i}.txt"))
        # a Booster reloaded from disk has no .best_iteration of its own - restore it
        m.best_iteration = meta["best_iterations"][i]
        if i == meta.get("focal_model_idx"):
            m._af_focal = True
        models.append(m)

    cnn_model = None
    cnn_path = os.path.join(weights_dir, "af_cnn_best.pt")
    if meta.get("use_cnn_in_blend") and os.path.exists(cnn_path):
        cnn_model = AFCNN(in_channels=len(AF_FEATURE_NAMES)).to(device)
        cnn_model.load_state_dict(torch.load(cnn_path, map_location=device))
        cnn_model.eval()

    return {
        "models": models,
        "cnn_model": cnn_model,
        "global_threshold": meta.get("global_threshold", 0.5),
        "per_landcover_thresholds": meta.get("per_landcover_thresholds"),
    }


In [ ]:
# === встроено из src/bs_pipeline.py - не редактировать здесь, правьте src/bs_pipeline.py и пересоберите через build_pipeline_notebook.py ===
"""BS module (burn contour + severity) inference pipeline: feature stacks,
siamese U-Net ensemble, TTA, thresholds, postprocessing.

Same logic as embedded in notebooks/02_bs_burn_severity.ipynb (via
build_notebooks.py) - kept here too as a plain importable module so
`inference.py` can run outside Colab/Jupyter using just a local directory of
weights. If you change feature engineering or model architecture here, mirror
the change in build_notebooks.py (and vice versa) - the two are not auto-synced.

Only inference-time code lives here. Training-only code (Dataset/augmentation,
the training loop, class weights, threshold search) stays in the notebook,
since training only ever happens on Colab.
"""
from __future__ import annotations

import json
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage as ndi

BS_S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8A", "B11", "B12"]
SCL_CLOUD_CLASSES = {3, 8, 9, 10}  # shadow / cloud medium+high probability / cirrus

BS_PRE_CHANNELS = BS_S2_BANDS + ["NDVI", "NBR", "NDMI", "MIRBI", "BAI", "GEMI", "VV_db", "VH_db", "RVI"]
BS_STATIC_CHANNELS = [
    "dNBR", "RdNBR", "RBR", "dVV", "dVH", "RBD", "LogRBR", "dRVI",
    "dem", "slope", "landcover", "cloud_mask",
]


def _refl(band):
    """Sentinel-2 L2A stores reflectance as uint16 * 10000."""
    return band.astype(np.float32) / 10000.0


def _ndvi(bands):
    red, nir = bands["B4"].astype(np.float32), bands["B8A"].astype(np.float32)
    return (nir - red) / (nir + red + 1e-6)


def _nbr(bands):
    b8a, b12 = bands["B8A"].astype(np.float32), bands["B12"].astype(np.float32)
    return (b8a - b12) / (b8a + b12 + 1e-6)


def _ndmi(bands):
    nir, swir1 = bands["B8A"].astype(np.float32), bands["B11"].astype(np.float32)
    return (nir - swir1) / (nir + swir1 + 1e-6)


def _mirbi(bands):
    swir1, swir2 = _refl(bands["B11"]), _refl(bands["B12"])
    return 10.0 * swir2 - 9.8 * swir1 + 2.0


def _bai(bands):
    red, nir = _refl(bands["B4"]), _refl(bands["B8A"])
    val = 1.0 / ((0.1 - red) ** 2 + (0.06 - nir) ** 2 + 1e-6)
    return np.clip(val, 0, 50)


def _gemi(bands):
    red, nir = _refl(bands["B4"]), _refl(bands["B8A"])
    eta = (2.0 * (nir ** 2 - red ** 2) + 1.5 * nir + 0.5 * red) / (nir + red + 0.5 + 1e-6)
    return eta * (1 - 0.25 * eta) - (red - 0.125) / (1 - red + 1e-6)


def _s1_db(band, lo=-30.0, hi=5.0):
    """int16 stores dB*100 (confirmed on real data) - convert to dB and clip to a
    physically sane range (radar shadow/layover outliers otherwise dominate
    normalization statistics)."""
    return np.clip(band.astype(np.float32) / 100.0, lo, hi)


def _rvi(vv_db, vh_db):
    """Radar Vegetation Index - defined for LINEAR power, not dB."""
    vv_lin, vh_lin = 10.0 ** (vv_db / 10.0), 10.0 ** (vh_db / 10.0)
    return 4.0 * vh_lin / (vv_lin + vh_lin + 1e-6)


def _build_date_branch(bands, s1):
    vv_db, vh_db = _s1_db(s1["VV"]), _s1_db(s1["VH"])
    stack = [bands[b].astype(np.float32) for b in BS_S2_BANDS] + [
        _ndvi(bands), _nbr(bands), _ndmi(bands), _mirbi(bands), _bai(bands), _gemi(bands),
        vv_db, vh_db, _rvi(vv_db, vh_db),
    ]
    return np.stack(stack, axis=0).astype(np.float32)


def build_bs_feature_stack_siamese(chip):
    """(pre_stack, post_stack, static_stack), shapes (18,H,W), (18,H,W), (12,H,W)."""
    s2pre, s2post = chip["s2_pre"], chip["s2_post"]
    s1pre, s1post = chip["s1_pre"], chip["s1_post"]
    aux = chip["aux"]

    pre_stack = _build_date_branch(s2pre, s1pre)
    post_stack = _build_date_branch(s2post, s1post)

    nbr_pre, nbr_post = _nbr(s2pre), _nbr(s2post)
    dnbr = nbr_pre - nbr_post
    rdnbr = np.clip(dnbr / (np.sqrt(np.abs(nbr_pre) / 1000.0) + 1e-6), -1000, 1000)
    rbr = np.clip(dnbr / (nbr_pre + 1.001), -3, 3)

    vv_pre_db, vh_pre_db = _s1_db(s1pre["VV"]), _s1_db(s1pre["VH"])
    vv_post_db, vh_post_db = _s1_db(s1post["VV"]), _s1_db(s1post["VH"])
    dvv = np.clip(vv_post_db - vv_pre_db, -20, 20)
    dvh = np.clip(vh_post_db - vh_pre_db, -20, 20)
    rbd = np.clip(vh_pre_db - vh_post_db, -20, 20)
    vh_pre_lin, vh_post_lin = 10.0 ** (vh_pre_db / 10.0), 10.0 ** (vh_post_db / 10.0)
    log_rbr = np.clip(np.log10((vh_pre_lin + 1e-6) / (vh_post_lin + 1e-6)), -3, 3)
    drvi = _rvi(vv_post_db, vh_post_db) - _rvi(vv_pre_db, vh_pre_db)

    cloud_mask = np.isin(s2post["SCL"], list(SCL_CLOUD_CLASSES)).astype(np.float32)
    static_stack = np.stack([
        dnbr, rdnbr, rbr,
        dvv, dvh, rbd, log_rbr, drvi,
        aux["dem"].astype(np.float32), aux["slope"].astype(np.float32),
        aux["landcover"].astype(np.float32), cloud_mask,
    ], axis=0).astype(np.float32)

    clean = lambda a: np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    return clean(pre_stack), clean(post_stack), clean(static_stack)


# ---------------------------------------------------------------------------
# Per-pixel feature stack for the LightGBM severity model.
#
# Why a per-pixel GBM at all, next to the siamese U-Net: burn severity is
# largely a per-pixel spectral problem (dNBR conditioned on land cover - the
# case statement itself says fixed dNBR scales fail because forest/steppe/
# cropland need different calibration). A GBM sees ~47M labelled pixels from
# 179 chips, while the U-Net sees 179 images - so the GBM is far less
# data-starved on exactly the part of Score that has the most headroom
# (mIoU_sev), and it is an order of magnitude cheaper at inference.
# ---------------------------------------------------------------------------

BS_PIXEL_FEATURE_NAMES = (
    [f"{b}_pre" for b in BS_S2_BANDS] + [f"{b}_post" for b in BS_S2_BANDS] + [
        "NDVI_pre", "NDVI_post", "dNDVI",
        "NBR_pre", "NBR_post", "dNBR", "RdNBR", "RBR",
        "NDMI_pre", "NDMI_post", "dNDMI",
        "MIRBI_pre", "MIRBI_post", "dMIRBI",
        "BAI_pre", "BAI_post", "dBAI",
        "GEMI_pre", "GEMI_post", "dGEMI",
        "VV_pre_db", "VH_pre_db", "VV_post_db", "VH_post_db",
        "dVV", "dVH", "RBD", "LogRBR",
        "RVI_pre", "RVI_post", "dRVI",
        "dem", "slope", "landcover", "cloud_mask",
        "dNBR_mean5", "dNBR_mean15", "dNBR_local_contrast",
        # --- scene-relative + texture block ---------------------------------
        # The case statement itself says a single dNBR scale does not transfer
        # between forest/steppe/cropland. Absolute dNBR is therefore a weak
        # feature across scenes; its rank/z-score WITHIN the chip is not, and
        # local std separates a real burn edge from sensor noise. Measured on
        # validation: IoU_burn 0.5880 -> 0.5891, mIoU_sev 0.5786 -> 0.5847,
        # with class-1 (light burn) IoU 0.376 -> 0.387.
        "dNBR_z", "dNBR_rank", "dNBR_med_rel",
        "NBR_post_z", "NBR_post_rank",
        "NDVI_post_z", "dNDVI_rank",
        "MIRBI_post_z", "dNBR_std5", "dNBR_std15", "dNBR_mean31",
        "NBR_post_mean5", "dNDVI_mean5", "dNBR_rank_mean15",
    ]
)
BS_PIXEL_LANDCOVER_IDX = BS_PIXEL_FEATURE_NAMES.index("landcover")


def _chip_z(x):
    """z-score within the chip - makes a spectral index scene-relative."""
    return (x - float(np.nanmean(x))) / (float(np.nanstd(x)) + 1e-6)


def _chip_rank(x):
    """percentile rank within the chip, 0..1 - fully scale-invariant."""
    flat = x.ravel()
    order = np.argsort(flat, kind="stable")
    r = np.empty(flat.shape[0], np.float32)
    r[order] = np.arange(flat.shape[0], dtype=np.float32) / max(flat.shape[0] - 1, 1)
    return r.reshape(x.shape)


def _std_filter(x, size):
    from scipy.ndimage import uniform_filter

    m = uniform_filter(x, size=size)
    return np.sqrt(np.maximum(uniform_filter(x * x, size=size) - m * m, 0.0))


def build_bs_pixel_features(chip):
    """(C, H, W) float32 in BS_PIXEL_FEATURE_NAMES order, for the LightGBM model.

    `dNBR_mean5`/`dNBR_mean15`/`dNBR_local_contrast` give the otherwise purely
    per-pixel model a bit of spatial context for free (a burnt pixel sits inside
    a burnt neighbourhood; an isolated low-dNBR pixel is usually noise)."""
    from scipy.ndimage import uniform_filter

    s2pre, s2post = chip["s2_pre"], chip["s2_post"]
    s1pre, s1post = chip["s1_pre"], chip["s1_post"]
    aux = chip["aux"]

    ndvi_pre, ndvi_post = _ndvi(s2pre), _ndvi(s2post)
    nbr_pre, nbr_post = _nbr(s2pre), _nbr(s2post)
    dnbr = nbr_pre - nbr_post
    rdnbr = np.clip(dnbr / (np.sqrt(np.abs(nbr_pre) / 1000.0) + 1e-6), -1000, 1000)
    rbr = np.clip(dnbr / (nbr_pre + 1.001), -3, 3)
    ndmi_pre, ndmi_post = _ndmi(s2pre), _ndmi(s2post)
    mirbi_pre, mirbi_post = _mirbi(s2pre), _mirbi(s2post)
    bai_pre, bai_post = _bai(s2pre), _bai(s2post)
    gemi_pre, gemi_post = _gemi(s2pre), _gemi(s2post)

    vv_pre_db, vh_pre_db = _s1_db(s1pre["VV"]), _s1_db(s1pre["VH"])
    vv_post_db, vh_post_db = _s1_db(s1post["VV"]), _s1_db(s1post["VH"])
    dvv = np.clip(vv_post_db - vv_pre_db, -20, 20)
    dvh = np.clip(vh_post_db - vh_pre_db, -20, 20)
    rbd = np.clip(vh_pre_db - vh_post_db, -20, 20)
    vh_pre_lin, vh_post_lin = 10.0 ** (vh_pre_db / 10.0), 10.0 ** (vh_post_db / 10.0)
    log_rbr = np.clip(np.log10((vh_pre_lin + 1e-6) / (vh_post_lin + 1e-6)), -3, 3)
    rvi_pre, rvi_post = _rvi(vv_pre_db, vh_pre_db), _rvi(vv_post_db, vh_post_db)

    dnbr_mean5 = uniform_filter(dnbr, size=5)
    dnbr_mean15 = uniform_filter(dnbr, size=15)

    dndvi = ndvi_post - ndvi_pre
    dnbr_rank = _chip_rank(dnbr)

    cloud_mask = np.isin(s2post["SCL"], list(SCL_CLOUD_CLASSES)).astype(np.float32)

    feats = np.stack(
        [s2pre[b].astype(np.float32) for b in BS_S2_BANDS]
        + [s2post[b].astype(np.float32) for b in BS_S2_BANDS]
        + [
            ndvi_pre, ndvi_post, ndvi_post - ndvi_pre,
            nbr_pre, nbr_post, dnbr, rdnbr, rbr,
            ndmi_pre, ndmi_post, ndmi_post - ndmi_pre,
            mirbi_pre, mirbi_post, mirbi_post - mirbi_pre,
            bai_pre, bai_post, bai_post - bai_pre,
            gemi_pre, gemi_post, gemi_post - gemi_pre,
            vv_pre_db, vh_pre_db, vv_post_db, vh_post_db,
            dvv, dvh, rbd, log_rbr,
            rvi_pre, rvi_post, rvi_post - rvi_pre,
            aux["dem"].astype(np.float32), aux["slope"].astype(np.float32),
            aux["landcover"].astype(np.float32), cloud_mask,
            dnbr_mean5, dnbr_mean15, dnbr - dnbr_mean15,
            _chip_z(dnbr), dnbr_rank, dnbr - float(np.nanmedian(dnbr)),
            _chip_z(nbr_post), _chip_rank(nbr_post),
            _chip_z(ndvi_post), _chip_rank(dndvi),
            _chip_z(mirbi_post), _std_filter(dnbr, 5), _std_filter(dnbr, 15),
            uniform_filter(dnbr, size=31),
            uniform_filter(nbr_post, size=5), uniform_filter(dndvi, size=5),
            uniform_filter(dnbr_rank, size=15),
        ],
        axis=0,
    ).astype(np.float32)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


# LightGBM stops walking the remaining trees for a pixel once the leading class
# is ahead by this margin. On a 512x512 chip almost every pixel is obviously
# unburnt, so this is where the time goes and where it can be given back:
# the model is 181 iterations x 4 classes = 724 trees over 262144 pixels.
#
# The margin was chosen by running the WHOLE BS pipeline (LightGBM -> refiner ->
# threshold -> postprocessing) on the validation split and scoring it with the
# organizers' own metric, not by eyeballing probabilities:
#
#   margin off -> 2.29 s/chip           IoU_burn 0.71323  mIoU_sev 0.67410
#   margin 7.0 -> 1.85 s/chip (1.24x)   IoU_burn 0.71323  mIoU_sev 0.67410
#   margin 5.0 -> 1.50 s/chip (1.53x)   IoU_burn 0.71339  mIoU_sev 0.67428
#   margin 3.5 -> 1.17 s/chip (1.95x)   IoU_burn 0.71320  mIoU_sev 0.67418
#
# At 3.5 the module's contribution to Score moves by +0.00001 - noise. (Pixel
# level, 0.038% of pixels across the test set do flip, but they are pixels the
# threshold was never going to be decided by, which is exactly why the metric
# does not move.)
#
# This matters for grading: the BS module stays CPU-bound on LightGBM even on
# the organizers' H100, because the GPU only removes the refiner. Halving the
# LightGBM time is what puts the whole inference inside the "under 5 minutes"
# bracket instead of on its boundary.
BS_LGB_EARLY_STOP_MARGIN = 3.5
BS_LGB_EARLY_STOP_FREQ = 10


def _lgb_predict(model, flat):
    """Predict with early stopping, falling back if the installed LightGBM does
    not accept those keywords - a speed optimisation must never be the reason
    inference fails on someone else's machine."""
    try:
        return model.predict(
            flat, num_iteration=model.best_iteration, pred_early_stop=True,
            pred_early_stop_freq=BS_LGB_EARLY_STOP_FREQ,
            pred_early_stop_margin=BS_LGB_EARLY_STOP_MARGIN)
    except (TypeError, ValueError):
        return model.predict(flat, num_iteration=model.best_iteration)


def predict_bs_proba_lgb(chip, model):
    """Full-image 4-class probabilities from the LightGBM model -> (4, H, W)."""
    feats = build_bs_pixel_features(chip)
    c, h, w = feats.shape
    probs = _lgb_predict(model, feats.reshape(c, -1).T)
    return probs.T.reshape(4, h, w).astype(np.float32)


# --------------------------------------------------------------------------
# Stacked spatial refiner
#
# The per-pixel LightGBM saturates around IoU_burn 0.59 / mIoU_sev 0.585 on
# validation, and the class it fails on is class 1 (light burn, IoU 0.39):
# spectrally light burn sits between unburnt and moderate burn, so a model that
# sees one pixel at a time cannot separate them. What does separate them is
# context - light burn forms rings around heavier burn, and fire scars are
# contiguous shapes. Hence a CNN that takes the LightGBM probability maps as
# input and refines them spatially, rather than replacing them.
#
# Two design choices make this safe rather than a gamble:
#   * ImageNet-pretrained ResNet34 encoder - 224 training chips is far too few
#     to learn low-level filters from scratch (the earlier from-scratch siamese
#     U-Net scored 0.274 against LightGBM's 0.374, i.e. it was data-starved);
#   * a residual connection onto log(P_lgb) with a ZERO-initialised output
#     layer, so at step 0 the network reproduces the LightGBM probabilities
#     exactly and training can only move away from them if that lowers the loss.
# --------------------------------------------------------------------------
BS_REFINER_FEATURE_NAMES = [
    "dNBR", "dNBR_rank", "dNBR_z", "dNBR_std5", "NBR_pre", "NBR_post",
    "dNDVI", "NDVI_post", "dNDMI", "dMIRBI", "dVV", "dVH",
    "dem", "slope", "landcover", "cloud_mask",
]
BS_REFINER_IDX = [BS_PIXEL_FEATURE_NAMES.index(n) for n in BS_REFINER_FEATURE_NAMES]
BS_REFINER_IN_CH = 4 + len(BS_REFINER_FEATURE_NAMES)


def build_refiner_input(pixel_feats, lgb_proba):
    """(BS_REFINER_IN_CH, H, W) float32: 4 LightGBM probability maps + a compact
    raw block, each raw channel z-scored WITHIN the chip so no normalisation
    statistics have to be stored alongside the weights."""
    raw = pixel_feats[BS_REFINER_IDX].astype(np.float32)
    out = np.empty((BS_REFINER_IN_CH, raw.shape[1], raw.shape[2]), np.float32)
    out[:4] = lgb_proba
    for i, name in enumerate(BS_REFINER_FEATURE_NAMES):
        if name == "landcover":
            out[4 + i] = raw[i] / 100.0            # categorical codes 10..100
        elif name == "cloud_mask":
            out[4 + i] = raw[i]                     # already 0/1
        else:
            out[4 + i] = _chip_z(raw[i])
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


class _Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = ConvBlock(cin + cskip, cout)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class BSRefiner(nn.Module):
    """ResNet34-U-Net that refines LightGBM probabilities. `pretrained=False` at
    inference time so constructing it never touches the network - the encoder
    weights come from the saved checkpoint like everything else."""

    def __init__(self, in_ch=BS_REFINER_IN_CH, n_classes=4, pretrained=False):
        super().__init__()
        import torchvision

        enc = torchvision.models.resnet34(
            weights="IMAGENET1K_V1" if pretrained else None)
        w = enc.conv1.weight.data.clone()
        enc.conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        if pretrained:
            reps = (in_ch + 2) // 3
            enc.conv1.weight.data = w.repeat(1, reps, 1, 1)[:, :in_ch] * (3.0 / in_ch)
        self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu)
        self.pool, self.l1, self.l2 = enc.maxpool, enc.layer1, enc.layer2
        self.l3, self.l4 = enc.layer3, enc.layer4
        self.d4, self.d3 = _Up(512, 256, 256), _Up(256, 128, 128)
        self.d2, self.d1 = _Up(128, 64, 64), _Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, n_classes, 1),
        )
        # identity start: logits = 0 + log(P_lgb)  =>  softmax == P_lgb
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x):
        lgb_logit = torch.log(x[:, :4].clamp_min(1e-6))
        s0 = self.stem(x)
        x1 = self.l1(self.pool(s0))
        x2, x3 = self.l2(x1), None
        x3 = self.l3(x2)
        x4 = self.l4(x3)
        y = self.d1(self.d2(self.d3(self.d4(x4, x3), x2), x1), s0)
        y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(y) + lgb_logit


def save_refiner_state(model, path):
    """Checkpoints go to disk in fp16: 98.5 MB -> 49.3 MB, which keeps them well
    under GitHub's hard 100 MB per-file limit. Measured cost of the round-trip:
    max |delta softmax| = 3.5e-6 and not a single pixel changes its argmax."""
    torch.save({k: (v.half() if v.is_floating_point() else v)
                for k, v in model.state_dict().items()}, path)


def load_refiner_state(model, path, device="cpu"):
    """Accepts both fp16 and fp32 checkpoints - always casts to the model's dtype."""
    sd = torch.load(path, map_location=device)
    model.load_state_dict({k: (v.float() if v.is_floating_point() else v)
                            for k, v in sd.items()})
    return model


@torch.no_grad()
def predict_bs_proba_refiner(refiner_input, models, device="cpu", use_tta=True):
    """Average softmax over the refiner ensemble (and flip-TTA) -> (4, H, W)."""
    x = torch.from_numpy(refiner_input).unsqueeze(0).to(device)
    variants = [(x, None)]
    if use_tta:
        variants += [(torch.flip(x, [-1]), [-1]), (torch.flip(x, [-2]), [-2]),
                     (torch.flip(x, [-1, -2]), [-1, -2])]
    total, n = 0.0, 0
    for model in models:
        for xv, dims in variants:
            p = torch.softmax(model(xv), dim=1)
            if dims is not None:
                p = torch.flip(p, dims)
            total = total + p
            n += 1
    return (total / n)[0].float().cpu().numpy()


class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SharedEncoder(nn.Module):
    def __init__(self, in_channels, base=32):
        super().__init__()
        self.input_norm = nn.BatchNorm2d(in_channels, affine=False)
        self.enc1 = ConvBlock(in_channels, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base * 8, base * 16)

    def forward(self, x):
        x = self.input_norm(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        return e1, e2, e3, e4, b


class SiameseUNet(nn.Module):
    """Shared-weight encoder for pre/post (change-invariant features, xView2
    1st/2nd place and BAM-CD/FLOGA pattern) + a static/diff stream fused once
    at the bottleneck."""

    def __init__(self, pair_channels, static_channels, n_classes, base=32):
        super().__init__()
        self.encoder = SharedEncoder(pair_channels, base)
        self.static_norm = nn.BatchNorm2d(static_channels, affine=False)
        self.static_stem = ConvBlock(static_channels, base * 16)
        self.bottleneck_fuse = nn.Conv2d(base * 16 * 3, base * 16, 1)
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 8 * 3, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 4 * 3, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 2 * 3, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 3, base)
        self.out_conv = nn.Conv2d(base, n_classes, 1)

    def forward(self, pre, post, static):
        e1p, e2p, e3p, e4p, bp = self.encoder(pre)
        e1o, e2o, e3o, e4o, bo = self.encoder(post)
        static_feat = F.adaptive_avg_pool2d(self.static_stem(self.static_norm(static)), bp.shape[-2:])
        b = self.bottleneck_fuse(torch.cat([bp, bo, static_feat], dim=1))
        d4 = self.dec4(torch.cat([self.up4(b), e4p, e4o], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3p, e3o], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2p, e2o], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1p, e1o], dim=1))
        return self.out_conv(d1)


def _tta_variants(pre, post, static):
    variants = []
    for flip_h, flip_v in [(False, False), (True, False), (False, True), (True, True)]:
        p, o, s = pre, post, static
        if flip_h:
            p, o, s = torch.flip(p, dims=[-1]), torch.flip(o, dims=[-1]), torch.flip(s, dims=[-1])
        if flip_v:
            p, o, s = torch.flip(p, dims=[-2]), torch.flip(o, dims=[-2]), torch.flip(s, dims=[-2])
        variants.append((p, o, s, flip_h, flip_v))
    return variants


def predict_bs_proba_tta(pre, post, static, models, device="cpu", use_tta=True):
    """Average softmax across the model ensemble and (optionally) TTA variants.
    pre/post/static are (C,H,W) tensors with no batch dim. Returns (4,H,W) numpy."""
    pre_b = pre.unsqueeze(0).to(device)
    post_b = post.unsqueeze(0).to(device)
    static_b = static.unsqueeze(0).to(device)
    variants = _tta_variants(pre_b, post_b, static_b) if use_tta else [(pre_b, post_b, static_b, False, False)]
    total = None
    with torch.no_grad():
        for model in models:
            for p, o, s, flip_h, flip_v in variants:
                probs = F.softmax(model(p, o, s), dim=1)
                if flip_h:
                    probs = torch.flip(probs, dims=[-1])
                if flip_v:
                    probs = torch.flip(probs, dims=[-2])
                total = probs if total is None else total + probs
    n = len(models) * len(variants)
    return (total / n).squeeze(0).cpu().numpy()


def apply_bs_thresholds(probs, burn_threshold=0.5, severity_weights=(1.0, 1.0, 1.0)):
    """"Is there burn at all" (P(1)+P(2)+P(3)) is thresholded separately from the
    severity argmax among classes 1/2/3 - IoU_burn (weight 0.35) and mIoU_sev
    (weight 0.30) are optimized somewhat independently in Score.

    `severity_weights` rescale the three severity probabilities before the
    argmax. mIoU_sev is a MEAN of per-class IoU, so a rare class is worth as
    much as a common one and is worth biasing towards; the weights are searched
    on validation by coordinate ascent."""
    p_burn = probs[1] + probs[2] + probs[3]
    is_burn = p_burn >= burn_threshold
    w = np.asarray(severity_weights, dtype=np.float32).reshape(3, 1, 1)
    severity = 1 + np.argmax(probs[1:4] * w, axis=0)
    return np.where(is_burn, severity, 0).astype(np.uint8)


def postprocess_bs_mask(pred, min_blob_size=20, dilate_classes=(2, 3), dilate_iters=1):
    """Drop small isolated false-positive blobs + class-specific dilation for 2/3
    (xView2 1st place pattern: network boundaries for the damaged class tend to
    be under-called at the edges)."""
    out = pred.copy()
    labeled, n = ndi.label(pred > 0)
    for i in range(1, n + 1):
        blob = labeled == i
        if blob.sum() < min_blob_size:
            out[blob] = 0
    for c in dilate_classes:
        class_mask = out == c
        if class_mask.any():
            dilated = ndi.binary_dilation(class_mask, iterations=dilate_iters)
            out[dilated & (out == 0)] = c
    return out


def try_slic_smooth(pred, guide_rgb, n_segments=300):
    """Optional SLIC superpixel majority-vote smoothing; silently returns pred
    unchanged if scikit-image isn't installed - must never break the pipeline."""
    try:
        from skimage.segmentation import slic
    except ImportError:
        return pred
    segments = slic(guide_rgb, n_segments=n_segments, compactness=10, start_label=0, channel_axis=-1)
    out = pred.copy()
    for seg_id in np.unique(segments):
        seg_mask = segments == seg_id
        values, counts = np.unique(pred[seg_mask], return_counts=True)
        out[seg_mask] = values[np.argmax(counts)]
    return out


def predict_bs_mask(chip_id, source, lgb_model=None, unet_models=None, refiner_models=None,
                     strategy="lgb", burn_threshold=0.76, min_blob_size=35,
                     severity_weights=(1.0, 1.0, 1.0), refiner_alpha=1.0,
                     device="cpu", use_tta=True):
    """Full BS pipeline for one chip -> uint8 mask (0..3). RLE-encoding per class
    (`submission_row(chip_id, c, mask == c)` for c in 1,2,3) is the caller's job.

    `strategy` is whatever won on validation in the training notebook:
    'lgb'     - per-pixel LightGBM alone,
    'refiner' - LightGBM probabilities refined by the stacked ResNet34-U-Net
                (blended with the raw LightGBM maps by `refiner_alpha`),
    'unet'    - the legacy siamese U-Net ensemble,
    'blend'   - mean of LightGBM and the legacy siamese U-Net."""
    chip = source.load_bs(chip_id)

    probs_lgb = None
    if strategy in ("lgb", "blend", "refiner"):
        pixel_feats = build_bs_pixel_features(chip)
        c, h, w = pixel_feats.shape
        probs_lgb = _lgb_predict(
            lgb_model, pixel_feats.reshape(c, -1).T
        ).T.reshape(4, h, w).astype(np.float32)

    if strategy == "refiner":
        refined = predict_bs_proba_refiner(
            build_refiner_input(pixel_feats, probs_lgb), refiner_models,
            device=device, use_tta=use_tta,
        )
        probs = refiner_alpha * refined + (1.0 - refiner_alpha) * probs_lgb
    elif strategy == "lgb":
        probs = probs_lgb
    else:
        pre, post, static = build_bs_feature_stack_siamese(chip)
        probs_unet = predict_bs_proba_tta(
            torch.from_numpy(pre), torch.from_numpy(post), torch.from_numpy(static),
            unet_models, device=device, use_tta=use_tta,
        )
        probs = probs_unet if strategy == "unet" else 0.5 * (probs_lgb + probs_unet)

    pred_mask = apply_bs_thresholds(probs, burn_threshold=burn_threshold,
                                     severity_weights=severity_weights)
    if min_blob_size:
        pred_mask = postprocess_bs_mask(pred_mask, min_blob_size=min_blob_size, dilate_classes=())
    return pred_mask


def save_bs_artifacts(weights_dir, models, burn_threshold, extra_meta=None):
    """Mirrors exactly what the training notebook writes per-model to
    CHECKPOINT_DIR - a weights folder saved this way can be copied straight into
    `weights/bs/` for `inference.py`."""
    os.makedirs(weights_dir, exist_ok=True)
    for i, m in enumerate(models):
        torch.save(m.state_dict(), os.path.join(weights_dir, f"bs_unet_{i}_best.pt"))
    meta = {
        "n_models": len(models),
        "pre_channels": list(BS_PRE_CHANNELS),
        "static_channels": list(BS_STATIC_CHANNELS),
        "burn_threshold": float(burn_threshold),
    }
    if extra_meta:
        meta.update(extra_meta)
    with open(os.path.join(weights_dir, "bs_unet_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)


def load_bs_artifacts(weights_dir, device="cpu"):
    """Loads whatever the training notebook produced, in the layout it writes:

        bs_lightgbm.txt + bs_lightgbm_meta.json   (per-pixel LightGBM, the default model)
        bs_strategy.json                          (strategy + burn_threshold + min_blob_size)
        bs_unet_{i}_best.pt + bs_unet_meta.json   (optional siamese U-Net ensemble)

    Returns a dict ready to splat into `predict_bs_mask`."""
    strategy, burn_threshold, min_blob_size = "lgb", 0.76, 35
    severity_weights, refiner_alpha = (1.0, 1.0, 1.0), 1.0
    strategy_path = os.path.join(weights_dir, "bs_strategy.json")
    if os.path.exists(strategy_path):
        with open(strategy_path, encoding="utf-8") as f:
            sj = json.load(f)
        strategy = sj.get("strategy", strategy)
        burn_threshold = sj.get("burn_threshold", burn_threshold)
        min_blob_size = sj.get("min_blob_size", min_blob_size)
        severity_weights = tuple(sj.get("severity_weights", severity_weights))
        refiner_alpha = sj.get("refiner_alpha", refiner_alpha)

    lgb_model = None
    lgb_path = os.path.join(weights_dir, "bs_lightgbm.txt")
    lgb_meta_path = os.path.join(weights_dir, "bs_lightgbm_meta.json")
    if os.path.exists(lgb_path):
        import lightgbm as lgb  # local import: only needed at inference time

        # NOT lgb.Booster(model_file=...): that opens the file inside LightGBM's
        # native code, which on Windows uses the system codepage and fails on any
        # non-ASCII path ("Could not open ...") even when the file is readable.
        with open(lgb_path, encoding="utf-8") as f:
            lgb_model = lgb.Booster(model_str=f.read())
        if os.path.exists(lgb_meta_path):
            with open(lgb_meta_path, encoding="utf-8") as f:
                # a Booster reloaded from disk has no .best_iteration of its own
                lgb_model.best_iteration = json.load(f)["best_iteration"]

    unet_models = None
    unet_meta_path = os.path.join(weights_dir, "bs_unet_meta.json")
    if strategy in ("unet", "blend") and os.path.exists(unet_meta_path):
        with open(unet_meta_path, encoding="utf-8") as f:
            unet_meta = json.load(f)
        base = unet_meta.get("base", 32)
        unet_models = []
        for i in range(unet_meta["n_models"]):
            m = SiameseUNet(pair_channels=len(BS_PRE_CHANNELS), static_channels=len(BS_STATIC_CHANNELS),
                             n_classes=4, base=base).to(device)
            m.load_state_dict(torch.load(os.path.join(weights_dir, f"bs_unet_{i}_best.pt"), map_location=device))
            m.eval()
            unet_models.append(m)

    refiner_models = None
    refiner_meta_path = os.path.join(weights_dir, "bs_refiner_meta.json")
    if strategy == "refiner" and os.path.exists(refiner_meta_path):
        with open(refiner_meta_path, encoding="utf-8") as f:
            refiner_meta = json.load(f)
        refiner_models = []
        for i in range(refiner_meta["n_models"]):
            m = BSRefiner(in_ch=refiner_meta.get("in_ch", BS_REFINER_IN_CH),
                           n_classes=4, pretrained=False).to(device)
            load_refiner_state(m, os.path.join(weights_dir, f"bs_refiner_{i}_best.pt"),
                                device=device)
            m.eval()
            refiner_models.append(m)

    if strategy == "refiner" and not refiner_models:
        raise FileNotFoundError(
            f"bs_strategy.json просит стратегию 'refiner', но в {weights_dir} нет "
            "bs_refiner_meta.json / bs_refiner_*_best.pt - положите их туда из CHECKPOINT_DIR."
        )
    if lgb_model is None and not unet_models:
        raise FileNotFoundError(
            f"В {weights_dir} нет ни bs_lightgbm.txt, ни весов сиамского U-Net - "
            "положите туда артефакты из CHECKPOINT_DIR после обучения."
        )

    return {
        "lgb_model": lgb_model,
        "unet_models": unet_models,
        "refiner_models": refiner_models,
        "strategy": strategy,
        "burn_threshold": burn_threshold,
        "min_blob_size": min_blob_size,
        "severity_weights": severity_weights,
        "refiner_alpha": refiner_alpha,
    }


### Источники чипов и разбиение

In [ ]:
def _resolve_source(dataset_dir, kind):
    """Kaggle то распаковывает .tar, то показывает его «развёрнутым» в браузере
    файлов, не распаковав на диск - поэтому проверяем оба варианта."""
    tar_path = os.path.join(dataset_dir, f'fire-{kind}-renamed.tar')
    if os.path.isfile(tar_path):
        print(f'источник {kind}: tar-архив -> {tar_path}')
        return TarChipSource(tar_path, root=kind)
    for candidate in (os.path.join(dataset_dir, f'fire-{kind}-renamed', kind),
                       os.path.join(dataset_dir, kind)):
        if os.path.isdir(candidate):
            print(f'источник {kind}: распакованная папка -> {candidate}')
            return DirChipSource(candidate)
    raise FileNotFoundError(f'не нашли данные {kind} в {dataset_dir}')


def resolve_train_source(dataset_dir):
    return _resolve_source(dataset_dir, 'train')


def resolve_test_source(dataset_dir):
    return _resolve_source(dataset_dir, 'test')


src = resolve_train_source(DATASET_DIR)
af_meta, bs_meta = src.meta('af'), src.meta('bs')
print('AF чипов:', len(af_meta), '| BS чипов:', len(bs_meta))

Разбиение — **по группам**, иначе метрика на валидации завышена:

* AF: `fire_event_id` во всех строках пустой, но 420 чипов приходятся всего на
  78 уникальных пролётов (`acq_datetime`) — соседние чипы одного пролёта это
  один и тот же пожар, разрезанный на куски. Группируем по пролёту.
* BS: группируем по `fire_event_id` — там он заполнен.

In [ ]:
def grouped_split(meta, group_col, val_frac=0.2, seed=RANDOM_SEED):
    groups = meta.set_index('chip_id')[group_col]
    uniq = sorted(groups.dropna().unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(uniq)
    val_groups = set(uniq[: max(1, int(val_frac * len(uniq)))])
    ids = meta['chip_id'].tolist()
    train = [c for c in ids if groups.get(c) not in val_groups]
    val = [c for c in ids if groups.get(c) in val_groups]
    return train, val, groups


af_train_ids, af_val_ids, af_groups = grouped_split(af_meta, 'acq_datetime')
bs_train_ids, bs_val_ids, bs_groups = grouped_split(bs_meta, 'fire_event_id')
print(f'AF: {len(af_train_ids)} train / {len(af_val_ids)} val '
      f'(пролётов: {af_meta["acq_datetime"].nunique()})')
print(f'BS: {len(bs_train_ids)} train / {len(bs_val_ids)} val '
      f'(событий: {bs_meta["fire_event_id"].nunique()})')

## 4. Модуль AF — активное горение

Метрика `F1_af` (вес 0.35 в Score) уже на 0.96, то есть запас тут ~0.013 —
почти всё, что можно, выжато. Поэтому здесь ничего не переусложняем:
ансамбль LightGBM из 3 сидов (один с focal loss) на 28 признаках, подбор
глобального порога и порогов по типам подстилающей поверхности.

CNN для AF **выключена намеренно**: на реальном прогоне она дала F1 0.41
против 0.96 у ансамбля, а max-блендинг с ней ронял итог до 0.67.

### Обучающие вспомогательные функции AF

Живут в ноутбуке, а не в `src/af_pipeline.py`: `inference.py` их не вызывает,
а в `src/` лежит только то, что нужно на инференсе.

In [ ]:
def focal_loss_lgb(y_pred, y_true, alpha=0.25, gamma=2.0, eps=1e-6):
    """grad/hess бинарного focal loss по сырому скору, центральными разностями -
    чтобы не выводить аналитическую формулу руками и не ошибиться молча."""
    def _value(x, t):
        p = np.clip(1.0 / (1.0 + np.exp(-x)), 1e-7, 1 - 1e-7)
        pt = np.where(t == 1, p, 1 - p)
        alpha_t = np.where(t == 1, alpha, 1 - alpha)
        return -alpha_t * (1 - pt) ** gamma * np.log(pt)

    grad = (_value(y_pred + eps, y_true) - _value(y_pred - eps, y_true)) / (2 * eps)
    hess = (_value(y_pred + eps, y_true) - 2 * _value(y_pred, y_true)
            + _value(y_pred - eps, y_true)) / (eps ** 2)
    return grad, np.clip(hess, 1e-6, None)


def build_af_training_table(source, chip_ids, win=21, neg_per_pos=40, hard_neg_frac=0.5,
                             seed=RANDOM_SEED, desc='AF table'):
    """Позитивы берём все (их 0.035% пикселей), негативы сэмплируем, причём
    половину - самые ТРУДНЫЕ: с наибольшим z_I4, то есть яркие в SWIR без пожара
    (нагретая почва, пески, факелы). На случайных негативах модель научилась бы
    отделять огонь от воды, а не от того, что реально даёт ложные срабатывания."""
    rng = np.random.default_rng(seed)
    X_parts, y_parts = [], []
    for chip_id in tqdm(list(chip_ids), desc=desc, leave=False):
        chip = source.load_af(chip_id)
        feats = extract_af_features(chip, win=win)
        mask = chip['mask']
        valid = nodata_to_ignore(mask, chip['mask_nodata']) & (chip['viirs']['valid'] > 0)
        pos_idx = np.flatnonzero((mask == 1) & valid)
        neg_idx_all = np.flatnonzero((mask == 0) & valid)
        if len(pos_idx) == 0 or len(neg_idx_all) == 0:
            n_take = min(len(neg_idx_all), 200)
            neg_idx = rng.choice(neg_idx_all, size=n_take, replace=False) if n_take > 0 else neg_idx_all
        else:
            z_I4_flat = feats[AF_FEATURE_NAMES.index('z_I4_w21')].ravel()[neg_idx_all]
            n_hard = min(len(neg_idx_all), int(len(pos_idx) * neg_per_pos * hard_neg_frac))
            hard_idx = neg_idx_all[np.argsort(-z_I4_flat)[:n_hard]]
            remaining = np.setdiff1d(neg_idx_all, hard_idx)
            n_rand = min(len(remaining), int(len(pos_idx) * neg_per_pos * (1 - hard_neg_frac)))
            rand_idx = (rng.choice(remaining, size=n_rand, replace=False)
                         if n_rand > 0 else np.array([], dtype=int))
            neg_idx = np.concatenate([hard_idx, rand_idx]).astype(int)
        idx = np.concatenate([pos_idx, neg_idx]).astype(int)
        X_parts.append(feats.reshape(feats.shape[0], -1)[:, idx].T)
        y_parts.append(mask.ravel()[idx])
    return np.concatenate(X_parts, axis=0), np.concatenate(y_parts, axis=0)


def sweep_af_threshold(source, chip_ids, proba_fn, thresholds=np.linspace(0.05, 0.95, 19)):
    """Порог подбирается на ПОЛНЫХ изображениях, а не на сэмплированной таблице:
    F1_af считается микро-усреднением по всем пикселям, а в таблице соотношение
    позитивов и негативов искусственное, то есть оптимум смещён."""
    chip_ids = list(chip_ids)
    cached = []
    for chip in tqdm(iter_chips(source, 'af', chip_ids), total=len(chip_ids),
                      desc='AF: кэшируем вероятности', leave=False):
        cached.append((proba_fn(chip), chip['mask'],
                        nodata_to_ignore(chip['mask'], chip['mask_nodata']),
                        chip['aux']['landcover']))
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        acc = ConfusionAccumulator(classes=(1,))
        for proba, mask, valid, _ in cached:
            acc.update((proba >= t).astype(np.uint8), mask, valid=valid)
        f1 = f1_af(acc)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1, cached


def calibrate_af_thresholds_by_landcover(cached, global_threshold,
                                          thresholds=np.linspace(0.05, 0.95, 19)):
    codes = np.unique(np.concatenate([lc.ravel() for _, _, _, lc in cached]))
    codes = codes[~np.isnan(codes)]   # landcover в AUX - float32, на краях бывает NaN
    per_landcover = {}
    for code in codes:
        best_t, best_f1 = float(global_threshold), -1.0
        for t in thresholds:
            acc = ConfusionAccumulator(classes=(1,))
            for proba, mask, valid, lc in cached:
                sel = valid & (lc == code)
                if sel.any():
                    acc.update((proba >= t).astype(np.uint8), mask, valid=sel)
            f1 = f1_af(acc)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        per_landcover[int(code)] = best_t
    return per_landcover


def f1_with_thresholds(cached, global_threshold, per_landcover):
    acc = ConfusionAccumulator(classes=(1,))
    for proba, mask, valid, lc in cached:
        acc.update(apply_af_thresholds(proba, lc, global_threshold, per_landcover).astype(np.uint8),
                    mask, valid=valid)
    return f1_af(acc)


_Xs, _ys = build_af_training_table(src, af_train_ids[:3], desc='AF smoke')
print('smoke test:', _Xs.shape, 'позитивов:', int(_ys.sum()), 'NaN:', int(np.isnan(_Xs).sum()))

In [ ]:
AF_N_SEEDS = 3
AF_ROUNDS = 2000

with stage('2. AF: таблицы признаков'):
    X_af_tr, y_af_tr = build_af_training_table(src, af_train_ids, desc='AF train')
    X_af_va, y_af_va = build_af_training_table(src, af_val_ids, desc='AF val')
print('AF таблицы:', X_af_tr.shape, X_af_va.shape, '| позитивов train:', int(y_af_tr.sum()))

In [ ]:
with stage('3. AF: обучение ансамбля'):
    af_models, af_paths = [], []
    for seed_i in range(AF_N_SEEDS):
        use_focal = seed_i == AF_N_SEEDS - 1   # последний сид - другой objective,
                                                # чтобы ошибки ансамбля были разными
        train_set = lgb.Dataset(X_af_tr, label=y_af_tr, feature_name=AF_FEATURE_NAMES,
                                 categorical_feature=[AF_LANDCOVER_IDX])
        val_set = lgb.Dataset(X_af_va, label=y_af_va, feature_name=AF_FEATURE_NAMES,
                               categorical_feature=[AF_LANDCOVER_IDX], reference=train_set)
        params = dict(metric=['auc', 'binary_logloss'], num_leaves=63, learning_rate=0.05,
                       feature_fraction=0.8 + 0.05 * (seed_i % 3),
                       bagging_fraction=0.8 + 0.05 * (seed_i % 2), bagging_freq=1,
                       min_data_in_leaf=50, seed=RANDOM_SEED + seed_i, verbose=-1,
                       num_threads=os.cpu_count())
        if use_focal:
            # LightGBM 4.x убрал аргумент fobj= у train() - кастомный objective
            # передаётся самим callable в params['objective']
            params['objective'] = lambda preds, data: focal_loss_lgb(preds, data.get_label())
        else:
            params['objective'] = 'binary'
            params['is_unbalance'] = True
        m = lgb.train(params, train_set, num_boost_round=AF_ROUNDS, valid_sets=[val_set],
                       callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(400)])
        if use_focal:
            m._af_focal = True   # objective='none' отдаёт сырой логит, сигмоида вручную
        p = os.path.join(CHECKPOINT_DIR, f'af_lightgbm_{seed_i}.txt')
        m.save_model(p)
        af_models.append(m)
        af_paths.append(p)
        print(f'  сид {seed_i+1}/{AF_N_SEEDS} (focal={use_focal}) best_iter={m.best_iteration}')

    _re = lgb.Booster(model_file=af_paths[0])
    assert np.allclose(af_models[0].predict(X_af_va[:200], num_iteration=af_models[0].best_iteration),
                       _re.predict(X_af_va[:200], num_iteration=af_models[0].best_iteration)), \
        'модель с диска не воспроизводит предсказания!'
    print('проверка воспроизводимости пройдена')

### Пороги AF

Дополнительно проверяем, помогают ли **пороги по landcover**: разные типы
покрова дают разный уровень ложных срабатываний (голая почва и пески греются
в SWIR без всякого пожара). Берём их, **только если они выигрывают на
валидации** — на 84 валидационных чипах редкие классы покрова представлены
единицами чипов, и порог для них легко переобучить.

In [ ]:
with stage('4. AF: калибровка порогов'):
    af_threshold, af_f1_global, af_cache = sweep_af_threshold(
        src, af_val_ids, lambda chip: predict_af_proba_ensemble(chip, af_models))
    print(f'глобальный порог: {af_threshold:.3f}  F1_af={af_f1_global:.4f}')

    af_lc_thresholds = calibrate_af_thresholds_by_landcover(af_cache, af_threshold)
    af_f1_lc = f1_with_thresholds(af_cache, af_threshold, af_lc_thresholds)
    print(f'пороги по landcover: F1_af={af_f1_lc:.4f}\n  {af_lc_thresholds}')

    if af_f1_lc > af_f1_global:
        af_f1_final = af_f1_lc
        print('ВЫБРАНО: пороги по landcover')
    else:
        af_f1_final, af_lc_thresholds = af_f1_global, {}
        print('ВЫБРАНО: один глобальный порог (по landcover не помогли)')

    with open(os.path.join(CHECKPOINT_DIR, 'af_lightgbm_meta.json'), 'w', encoding='utf-8') as f:
        json.dump({'feature_names': AF_FEATURE_NAMES,
                   'categorical_feature_idx': [AF_LANDCOVER_IDX],
                   'n_seeds': AF_N_SEEDS, 'focal_model_idx': AF_N_SEEDS - 1,
                   'best_iterations': [m.best_iteration for m in af_models],
                   'global_threshold': float(af_threshold),
                   'per_landcover_thresholds': {str(k): float(v) for k, v in af_lc_thresholds.items()},
                   'use_cnn_in_blend': False, 'f1_val': float(af_f1_final),
                   'seed': RANDOM_SEED, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')},
                  f, ensure_ascii=False, indent=2)
print(f'ИТОГ AF: F1_af={af_f1_final:.4f}  ->  вклад в Score {0.35*af_f1_final:.4f} из 0.35')
del af_cache

## 5. Модуль BS — контур гари и степень поражения

Здесь весь оставшийся запас: `IoU_burn` и `mIoU_sev` дают 0.65 Score, а
набрано пока ~0.37. Базовая модель — **попиксельный LightGBM на 70 признаках**.

Почему не сегментационная сеть как основная: 224 обучающих чипа для сети
мало (обучавшийся с нуля сиамский U-Net дал 0.274 против 0.374 у бустинга),
а бустинг видит ~47 млн размеченных пикселей из тех же чипов.

**Новое в наборе признаков** — блок «относительно сцены»: сама постановка
кейса говорит, что единая шкала dNBR не переносится между лесом, степью и
пашней. Абсолютный dNBR поэтому слабый признак, а его ранг и z-оценка
**внутри чипа** — сильный. Плюс локальное СКО, отделяющее настоящий край
гари от шума. Замер на валидации: `IoU_burn` 0.588→0.589, `mIoU_sev`
0.579→0.585, и главное — IoU слабого класса 1 вырос 0.376→0.387.

In [ ]:
BS_LGB_MAX_PER_CLASS = 5000   # пикселей каждого класса с чипа
BS_LGB_BOUNDARY_EXTRA = 3000  # доп. пиксели у границы гари
BS_LGB_NUM_LEAVES = 127
BS_LGB_ROUNDS = 700
BS_LGB_LR = 0.05


def build_bs_training_table(source, chip_ids, max_per_class=BS_LGB_MAX_PER_CLASS,
                             boundary_extra=BS_LGB_BOUNDARY_EXTRA,
                             seed=RANDOM_SEED, desc='BS table'):
    """Стратифицированная выборка по классам + дополнительная выборка у границы.

    Случайные пиксели класса 0 почти все лежат далеко от контура, то есть модель
    тренируется на лёгких примерах, а IoU_burn решается именно на контуре. Это
    тот же приём, что hard negative mining в модуле AF.
    Замер: вклад модуля 0.3660 -> 0.3737."""
    rng = np.random.default_rng(seed)
    X_parts, y_parts = [], []
    for chip_id in tqdm(list(chip_ids), desc=desc, leave=False):
        chip = source.load_bs(chip_id)
        feats = build_bs_pixel_features(chip)
        mask = chip['mask']
        valid = nodata_to_ignore(mask, chip['mask_nodata'])
        flat = feats.reshape(feats.shape[0], -1)
        mask_flat, valid_flat = mask.ravel(), valid.ravel()

        for c in range(4):
            idx = np.flatnonzero((mask_flat == c) & valid_flat)
            if len(idx) == 0:
                continue
            take = min(len(idx), max_per_class)
            sel = rng.choice(idx, size=take, replace=False)
            X_parts.append(flat[:, sel].T)
            y_parts.append(np.full(take, c, dtype=np.int32))

        burn = mask > 0
        if boundary_extra and burn.any():
            outer_ring = ndi.binary_dilation(burn, iterations=6) & (~burn)
            inner_band = burn & (~ndi.binary_erosion(burn, iterations=6))
            for zone in (outer_ring, inner_band):
                zone_idx = np.flatnonzero(zone.ravel() & valid_flat)
                if len(zone_idx) == 0:
                    continue
                take = min(len(zone_idx), boundary_extra)
                sel = rng.choice(zone_idx, size=take, replace=False)
                X_parts.append(flat[:, sel].T)
                y_parts.append(mask_flat[sel].astype(np.int32))
    return np.concatenate(X_parts, axis=0), np.concatenate(y_parts, axis=0)


BS_LGB_PARAMS = dict(
    objective='multiclass', num_class=4, metric='multi_logloss',
    learning_rate=BS_LGB_LR, num_leaves=BS_LGB_NUM_LEAVES, min_data_in_leaf=100,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=os.cpu_count(), verbose=-1, seed=RANDOM_SEED,
)
_LC = BS_PIXEL_LANDCOVER_IDX


def train_bs_lgb(X_tr, y_tr, X_va, y_va, rounds=BS_LGB_ROUNDS):
    return lgb.train(
        BS_LGB_PARAMS,
        lgb.Dataset(X_tr, y_tr, feature_name=BS_PIXEL_FEATURE_NAMES, categorical_feature=[_LC]),
        num_boost_round=rounds,
        valid_sets=[lgb.Dataset(X_va, y_va, feature_name=BS_PIXEL_FEATURE_NAMES,
                                 categorical_feature=[_LC])],
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(100)],
    )


print('признаков BS:', len(BS_PIXEL_FEATURE_NAMES))

In [ ]:
with stage('5. BS: обучение LightGBM'):
    X_bs_tr, y_bs_tr = build_bs_training_table(src, bs_train_ids, desc='BS train')
    X_bs_va, y_bs_va = build_bs_training_table(src, bs_val_ids, desc='BS val')
    print('таблицы:', X_bs_tr.shape, X_bs_va.shape,
          '| классы train:', np.bincount(y_bs_tr, minlength=4))

    bs_lgb_model = train_bs_lgb(X_bs_tr, y_bs_tr, X_bs_va, y_bs_va)
    print('best_iteration:', bs_lgb_model.best_iteration)
    _imp = sorted(zip(BS_PIXEL_FEATURE_NAMES, bs_lgb_model.feature_importance('split')),
                   key=lambda t: -t[1])[:15]
    print('топ-15 признаков:', [f'{n}:{v}' for n, v in _imp])

    bs_lgb_path = os.path.join(CHECKPOINT_DIR, 'bs_lightgbm.txt')
    bs_lgb_model.save_model(bs_lgb_path)
    with open(os.path.join(CHECKPOINT_DIR, 'bs_lightgbm_meta.json'), 'w', encoding='utf-8') as f:
        json.dump({'feature_names': BS_PIXEL_FEATURE_NAMES,
                   'categorical_feature_idx': [_LC], 'params': {k: str(v) for k, v in BS_LGB_PARAMS.items()},
                   'best_iteration': bs_lgb_model.best_iteration,
                   'max_per_class': BS_LGB_MAX_PER_CLASS,
                   'boundary_extra': BS_LGB_BOUNDARY_EXTRA, 'seed': RANDOM_SEED,
                   'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}, f, ensure_ascii=False, indent=2)

    _re = lgb.Booster(model_file=bs_lgb_path)
    assert np.allclose(_re.predict(X_bs_va[:200], num_iteration=bs_lgb_model.best_iteration),
                       bs_lgb_model.predict(X_bs_va[:200], num_iteration=bs_lgb_model.best_iteration)), \
        'модель с диска не воспроизводит предсказания!'
    print('сохранено и проверено:', bs_lgb_path)
del X_bs_tr, y_bs_tr

## 6. OOF-карты вероятностей для стекинга

Уточнитель из секции 7 получает на вход вероятности LightGBM. Если подать ему
предсказания модели, которая эти же чипы видела при обучении, он научится
слепо им доверять — на тесте, где вероятности заметно хуже, это развалится.

Поэтому обучающие карты делаются **out-of-fold**: обучающие чипы режутся на
`BS_OOF_FOLDS` групп по `fire_event_id`, для каждой группы вероятности даёт
модель, эту группу не видевшая. Ровно так же, как на настоящем тесте.

In [ ]:
BS_OOF_FOLDS = 3
BS_OOF_ROUNDS = 400   # фолдам полная глубина не нужна, они только для входа сети

with stage('6. BS: OOF-карты для стекинга'):
    _tr_groups = np.array([bs_groups.get(c) for c in bs_train_ids])
    _uniq = sorted(set(_tr_groups))
    _rng = np.random.default_rng(RANDOM_SEED)
    _rng.shuffle(_uniq)
    _fold_of = {g: i % BS_OOF_FOLDS for i, g in enumerate(_uniq)}

    oof_proba = {}
    for fold in range(BS_OOF_FOLDS):
        fit_ids = [c for c in bs_train_ids if _fold_of[bs_groups.get(c)] != fold]
        out_ids = [c for c in bs_train_ids if _fold_of[bs_groups.get(c)] == fold]
        print(f'  фолд {fold+1}/{BS_OOF_FOLDS}: обучение на {len(fit_ids)}, OOF для {len(out_ids)}')
        Xf, yf = build_bs_training_table(src, fit_ids, desc=f'fold {fold} table')
        mf = train_bs_lgb(Xf, yf, X_bs_va, y_bs_va, rounds=BS_OOF_ROUNDS)
        del Xf, yf
        for chip_id in tqdm(out_ids, desc=f'fold {fold} OOF', leave=False):
            chip = src.load_bs(chip_id)
            feats = build_bs_pixel_features(chip)
            c, h, w = feats.shape
            p = mf.predict(feats.reshape(c, -1).T, num_iteration=mf.best_iteration)
            # float16: 224 чипа x 4 x 512 x 512 float32 = 940 МБ, в half - 470 МБ
            oof_proba[chip_id] = p.T.reshape(4, h, w).astype(np.float16)
        del mf
    print('OOF готов для', len(oof_proba), 'чипов')

Для валидационных чипов вход уточнителя берётся от **основной** модели
(обученной на всех train-чипах) — это ровно та ситуация, что будет на тесте.

In [ ]:
with stage('7. BS: вероятности LightGBM на валидации'):
    val_lgb_proba, val_masks, val_valid = {}, {}, {}
    val_feats_cache = {}
    for chip_id in tqdm(bs_val_ids, desc='BS val proba'):
        chip = src.load_bs(chip_id)
        feats = build_bs_pixel_features(chip)
        c, h, w = feats.shape
        p = bs_lgb_model.predict(feats.reshape(c, -1).T,
                                  num_iteration=bs_lgb_model.best_iteration)
        val_lgb_proba[chip_id] = p.T.reshape(4, h, w).astype(np.float32)
        val_feats_cache[chip_id] = build_refiner_input(feats, val_lgb_proba[chip_id]).astype(np.float16)
        val_masks[chip_id] = chip['mask']
        val_valid[chip_id] = nodata_to_ignore(chip['mask'], chip['mask_nodata'])
print('валидационных карт:', len(val_lgb_proba))

## 7. Пространственный уточнитель (GPU)

Попиксельный бустинг упирается в `IoU_burn` ≈ 0.59 / `mIoU_sev` ≈ 0.585, и
проваливается он на классе 1 (слабое поражение, IoU 0.39): спектрально слабая
гарь лежит между несгоревшим и умеренным, и модель, видящая один пиксель,
их не разделит. Разделяет контекст — слабая гарь идёт каймой вокруг сильной,
а пожарища это связные фигуры.

Две вещи делают это не лотереей:

1. **Предобученный на ImageNet энкодер ResNet34.** 224 чипа — слишком мало,
   чтобы выучить низкоуровневые фильтры с нуля (что и показал прошлый
   сиамский U-Net: 0.274 против 0.374 у бустинга).
2. **Остаточная связь на `log(P_lgb)` с нулевой инициализацией выходного
   слоя.** На шаге 0 сеть выдаёт в точности вероятности LightGBM (проверено:
   расхождение 1.8e-7), то есть стартует не с нуля, а с текущего лучшего
   решения и может только уходить от него туда, где лосс ниже.

Плюс секция 8 всё равно сравнит уточнитель с голым LightGBM на валидации и
возьмёт лучшее — просадки быть не может по построению.

In [ ]:
BS_REFINER_EPOCHS = 30
BS_REFINER_CROP = 256
BS_REFINER_CROPS_PER_CHIP = 4
BS_REFINER_BATCH = 8
BS_REFINER_LR = 3e-4
BS_REFINER_N_MODELS = 1     # ансамбль; на H100 каждая модель ~2 с на весь тест
BS_REFINER_PRETRAINED = True


import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


class RefinerDataset(Dataset):
    """Случайные кропы + флипы. Кроп, целиком попавший в несгоревшую область,
    бесполезен - половину кропов принудительно центрируем на гари."""

    def __init__(self, chip_ids, inputs, masks, crop=BS_REFINER_CROP,
                  crops_per_chip=BS_REFINER_CROPS_PER_CHIP, seed=RANDOM_SEED):
        self.ids = list(chip_ids)
        self.inputs, self.masks = inputs, masks
        self.crop, self.n = crop, crops_per_chip
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.ids) * self.n

    def __getitem__(self, i):
        chip_id = self.ids[i // self.n]
        x, m = self.inputs[chip_id], self.masks[chip_id]
        H, W = m.shape
        c = self.crop
        burn = np.argwhere(m > 0)
        if len(burn) and self.rng.random() < 0.5:
            cy, cx = burn[self.rng.integers(len(burn))]
            y0 = int(np.clip(cy - c // 2, 0, max(H - c, 0)))
            x0 = int(np.clip(cx - c // 2, 0, max(W - c, 0)))
        else:
            y0 = int(self.rng.integers(0, max(H - c, 0) + 1))
            x0 = int(self.rng.integers(0, max(W - c, 0) + 1))
        xc = np.asarray(x[:, y0:y0 + c, x0:x0 + c], dtype=np.float32)
        mc = np.asarray(m[y0:y0 + c, x0:x0 + c], dtype=np.int64)
        if self.rng.random() < 0.5:
            xc, mc = xc[:, :, ::-1].copy(), mc[:, ::-1].copy()
        if self.rng.random() < 0.5:
            xc, mc = xc[:, ::-1, :].copy(), mc[::-1, :].copy()
        return torch.from_numpy(xc), torch.from_numpy(mc)


def squared_dice_loss(logits, target, n_classes=4, eps=1.0):
    probs = torch.softmax(logits, dim=1)
    onehot = F.one_hot(target, n_classes).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    inter = (probs * onehot).sum(dims)
    denom = (probs ** 2).sum(dims) + (onehot ** 2).sum(dims)
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def ordinal_penalty(logits, target):
    """Классы упорядочены 0<1<2<3: ошибка на две ступени хуже, чем на одну."""
    probs = torch.softmax(logits, dim=1)
    levels = torch.arange(probs.shape[1], device=probs.device).view(1, -1, 1, 1).float()
    expected = (probs * levels).sum(1)
    return F.l1_loss(expected, target.float())


# torch 2.4 объявил torch.cuda.amp устаревшим в пользу torch.amp с явным
# устройством; в Colab версия плавает, поэтому пробуем новый API и падаем
# обратно на старый
_AMP_ON = DEVICE == 'cuda'


def make_grad_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=_AMP_ON)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=_AMP_ON)


def autocast():
    try:
        return torch.amp.autocast('cuda', enabled=_AMP_ON)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=_AMP_ON)


print('уточнитель: вход', BS_REFINER_IN_CH, 'каналов ->', BS_REFINER_FEATURE_NAMES)

In [ ]:
with stage('8. BS: подготовка входов уточнителя'):
    train_inputs, train_masks = {}, {}
    for chip_id in tqdm(bs_train_ids, desc='refiner inputs'):
        chip = src.load_bs(chip_id)
        feats = build_bs_pixel_features(chip)
        p = oof_proba[chip_id].astype(np.float32)
        train_inputs[chip_id] = build_refiner_input(feats, p).astype(np.float16)
        train_masks[chip_id] = chip['mask']
    _mb = sum(v.nbytes for v in train_inputs.values()) / 1e6
    print(f'входов: {len(train_inputs)} чипов, {_mb:.0f} МБ в float16')
    del oof_proba

In [ ]:
def evaluate_refiner(models, alpha=1.0, burn_threshold=0.5, use_tta=False):
    """Вклад модуля BS в Score на валидации для данного набора моделей."""
    acc_b = ConfusionAccumulator(classes=(1,))
    acc_s = ConfusionAccumulator(classes=(1, 2, 3))
    for chip_id in bs_val_ids:
        x = np.asarray(val_feats_cache[chip_id], dtype=np.float32)
        p = predict_bs_proba_refiner(x, models, device=DEVICE, use_tta=use_tta)
        probs = alpha * p + (1 - alpha) * val_lgb_proba[chip_id]
        pred = apply_bs_thresholds(probs, burn_threshold=burn_threshold)
        acc_b.update(binarize_burn(pred), binarize_burn(val_masks[chip_id]), valid=val_valid[chip_id])
        acc_s.update(pred, val_masks[chip_id], valid=val_valid[chip_id])
    return 0.35 * iou_burn(acc_b) + 0.30 * miou_severity(acc_s), iou_burn(acc_b), miou_severity(acc_s)


with stage('9. BS: обучение уточнителя'):
    _counts = np.bincount(np.concatenate([m.ravel() for m in train_masks.values()]), minlength=4)
    # корень из обратной частоты, а не сама обратная частота: при inverse-freq
    # класс «не горело» получал вес 0.045 против 1.8 у горелых и сеть была
    # вынуждена видеть гарь почти везде
    _w = np.sqrt(_counts.sum() / (4.0 * np.maximum(_counts, 1)))
    class_weights = torch.tensor(_w / _w.mean(), dtype=torch.float32, device=DEVICE)
    print('частоты классов:', _counts, '| веса:', class_weights.cpu().numpy().round(3))

    refiner_models = []
    for mi in range(BS_REFINER_N_MODELS):
        torch.manual_seed(RANDOM_SEED + mi)
        model = BSRefiner(in_ch=BS_REFINER_IN_CH, n_classes=4,
                           pretrained=BS_REFINER_PRETRAINED).to(DEVICE)
        ds = RefinerDataset(bs_train_ids, train_inputs, train_masks, seed=RANDOM_SEED + mi)
        # num_workers=0: воркеры DataLoader на Colab/py3.13 сыплют AssertionError
        dl = DataLoader(ds, batch_size=BS_REFINER_BATCH, shuffle=True, num_workers=0, drop_last=True)
        opt = torch.optim.AdamW(model.parameters(), lr=BS_REFINER_LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=BS_REFINER_LR, total_steps=BS_REFINER_EPOCHS * len(dl), pct_start=0.3)
        scaler = make_grad_scaler()

        base_score = evaluate_refiner([model])[0]   # == голый LightGBM, тождественный старт
        print(f'модель {mi+1}: старт (== LightGBM) score={base_score:.4f}')
        best_score, best_path = base_score, os.path.join(CHECKPOINT_DIR, f'bs_refiner_{mi}_best.pt')
        save_refiner_state(model, best_path)   # fp16: 49 МБ вместо 99, лимит GitHub - 100

        for epoch in range(BS_REFINER_EPOCHS):
            model.train()
            running = 0.0
            for xb, yb in tqdm(dl, desc=f'модель {mi+1} эпоха {epoch+1}/{BS_REFINER_EPOCHS}', leave=False):
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with autocast():
                    logits = model(xb)
                    loss = (F.cross_entropy(logits, yb, weight=class_weights)
                            + squared_dice_loss(logits, yb)
                            + 0.2 * ordinal_penalty(logits, yb))
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                running += loss.item()
            model.eval()
            sc, ib, ms = evaluate_refiner([model])
            flag = ''
            if sc > best_score:
                best_score = sc
                save_refiner_state(model, best_path)
                flag = '  <-- лучшая, сохранена'
            print(f'  эпоха {epoch+1:02d}: loss={running/max(len(dl),1):.4f} '
                  f'score={sc:.4f} (IoU_burn={ib:.4f}, mIoU_sev={ms:.4f}){flag}', flush=True)

        load_refiner_state(model, best_path, device=DEVICE)
        model.eval()
        refiner_models.append(model)
        print(f'модель {mi+1}: лучший score={best_score:.4f} (старт был {base_score:.4f})')

    with open(os.path.join(CHECKPOINT_DIR, 'bs_refiner_meta.json'), 'w', encoding='utf-8') as f:
        json.dump({'n_models': len(refiner_models), 'in_ch': BS_REFINER_IN_CH,
                   'feature_names': BS_REFINER_FEATURE_NAMES,
                   'epochs': BS_REFINER_EPOCHS, 'crop': BS_REFINER_CROP,
                   'pretrained_encoder': BS_REFINER_PRETRAINED, 'seed': RANDOM_SEED},
                  f, ensure_ascii=False, indent=2)
del train_inputs, train_masks

## 8. Калибровка: стратегия, порог, веса классов, постобработка

Ничего не включается вслепую — каждый шаг проверяется на валидации и
принимается, только если поднимает вклад модуля в Score.

1. **Стратегия**: голый LightGBM против уточнителя, плюс перебор доли
   уточнителя `alpha` в смеси.
2. **Порог «есть гарь»** — отдельно от argmax степени: `IoU_burn` и
   `mIoU_sev` входят в Score независимыми слагаемыми.
3. **Веса классов степени**: `mIoU_sev` это СРЕДНЕЕ по классам, поэтому
   редкий класс стоит столько же, сколько частый, и его имеет смысл
   подтолкнуть.
4. **Удаление мелких блобов**.

In [ ]:
BS_TTA = True   # на H100 4 флипа стоят единицы секунд на весь тест

with stage('10. BS: калибровка'):
    cand_refined = {}
    for chip_id in tqdm(bs_val_ids, desc='refiner val proba'):
        x = np.asarray(val_feats_cache[chip_id], dtype=np.float32)
        cand_refined[chip_id] = predict_bs_proba_refiner(x, refiner_models, device=DEVICE, use_tta=BS_TTA)

    def eval_cfg(alpha, thr, weights=(1.0, 1.0, 1.0), min_blob=0):
        acc_b = ConfusionAccumulator(classes=(1,))
        acc_s = ConfusionAccumulator(classes=(1, 2, 3))
        for chip_id in bs_val_ids:
            probs = alpha * cand_refined[chip_id] + (1 - alpha) * val_lgb_proba[chip_id]
            pred = apply_bs_thresholds(probs, burn_threshold=thr, severity_weights=weights)
            if min_blob:
                pred = postprocess_bs_mask(pred, min_blob_size=min_blob, dilate_classes=())
            acc_b.update(binarize_burn(pred), binarize_burn(val_masks[chip_id]), valid=val_valid[chip_id])
            acc_s.update(pred, val_masks[chip_id], valid=val_valid[chip_id])
        ib, ms = iou_burn(acc_b), miou_severity(acc_s)
        return 0.35 * ib + 0.30 * ms, ib, ms, [acc_s.iou(c) for c in (1, 2, 3)]

    # 1) доля уточнителя в смеси + порог. alpha=0 - это голый LightGBM,
    #    то есть прошлое решение всегда в переборе как запасной вариант
    best = (-1.0, 0.0, 0.5)
    for a in (0.0, 0.25, 0.5, 0.75, 1.0):
        best_a = (-1.0, a, 0.5)
        for t in np.linspace(0.30, 0.90, 25):
            s = eval_cfg(a, float(t))[0]
            if s > best_a[0]:
                best_a = (s, a, float(t))
        print(f'  alpha={a:.2f}: лучший score={best_a[0]:.4f} при пороге {best_a[2]:.3f}')
        if best_a[0] > best[0]:
            best = best_a
    bs_score_val, refiner_alpha, burn_threshold = best
    print(f'ВЫБРАНО alpha={refiner_alpha:.2f}, порог={burn_threshold:.3f} -> score={bs_score_val:.4f}')

    # 2) веса классов степени (координатный подъём)
    severity_weights = [1.0, 1.0, 1.0]
    for _ in range(3):
        for k in range(3):
            for cand in (0.6, 0.75, 0.9, 1.0, 1.15, 1.35, 1.6, 2.0):
                trial = list(severity_weights)
                trial[k] = cand
                s = eval_cfg(refiner_alpha, burn_threshold, tuple(trial))[0]
                if s > bs_score_val:
                    bs_score_val, severity_weights = s, trial
        for t in np.linspace(max(0.2, burn_threshold - 0.12), min(0.95, burn_threshold + 0.12), 13):
            s = eval_cfg(refiner_alpha, float(t), tuple(severity_weights))[0]
            if s > bs_score_val:
                bs_score_val, burn_threshold = s, float(t)
    print(f'веса степени={severity_weights}, порог={burn_threshold:.3f} -> score={bs_score_val:.4f}')

    # 3) постобработка
    bs_min_blob_size = 0
    for mb in (10, 20, 35, 50, 80):
        s, ib, ms, per = eval_cfg(refiner_alpha, burn_threshold, tuple(severity_weights), mb)
        print(f'  min_blob={mb}: score={s:.4f} (IoU_burn={ib:.4f}, mIoU_sev={ms:.4f})')
        if s > bs_score_val:
            bs_score_val, bs_min_blob_size = s, mb

    bs_strategy = 'refiner' if refiner_alpha > 0 else 'lgb'
    s, ib, ms, per = eval_cfg(refiner_alpha, burn_threshold, tuple(severity_weights), bs_min_blob_size)
    print()
    print(f'ИТОГ BS: стратегия={bs_strategy} alpha={refiner_alpha} порог={burn_threshold:.3f} '
          f'веса={severity_weights} min_blob={bs_min_blob_size}')
    print(f'  IoU_burn={ib:.4f}  mIoU_sev={ms:.4f}  вклад={s:.4f} из 0.65')
    print(f'  IoU по классам степени: 1={per[0]:.3f}  2={per[1]:.3f}  3={per[2]:.3f}')

    with open(os.path.join(CHECKPOINT_DIR, 'bs_strategy.json'), 'w', encoding='utf-8') as f:
        json.dump({'strategy': bs_strategy, 'refiner_alpha': float(refiner_alpha),
                   'burn_threshold': float(burn_threshold),
                   'severity_weights': [float(v) for v in severity_weights],
                   'min_blob_size': int(bs_min_blob_size), 'use_tta': bool(BS_TTA),
                   'score_val': float(s), 'iou_burn_val': float(ib), 'miou_sev_val': float(ms)},
                  f, ensure_ascii=False, indent=2)

In [ ]:
TOTAL_VAL_SCORE = 0.35 * af_f1_final + bs_score_val
print('=' * 62)
print('ОЖИДАЕМЫЙ SCORE НА ВАЛИДАЦИИ')
print('=' * 62)
print(f'  0.35 * F1_af      = 0.35 * {af_f1_final:.4f} = {0.35*af_f1_final:.4f}')
print(f'  0.35 * IoU_burn   = 0.35 * {ib:.4f} = {0.35*ib:.4f}')
print(f'  0.30 * mIoU_sev   = 0.30 * {ms:.4f} = {0.30*ms:.4f}')
print(f'  ИТОГО             = {TOTAL_VAL_SCORE:.4f}')
print('=' * 62)
del cand_refined, val_feats_cache

## 9. Инференс на тесте — замеряемый блок

Ниже — ровно то, что будут запускать организаторы: модели поднимаются
**с диска** (как это делает `inference.py`), прогоняется весь тест, пишется
`submission.csv`. Замеряется отдельно и целиком, включая загрузку весов.

Тестовые чипы не имеют геопривязки и дат (в `test/meta.csv` только
`chip_id, kind, width, height, gsd, valid_frac, cloud_frac`), поэтому ни
восстановления координат, ни обращения к FIRMS/MCD64A1 по тестовым
территориям здесь нет и быть не может — это основания для дисквалификации.

In [ ]:
# веса кладём в ту же раскладку, что ждёт inference.py
WEIGHTS_DIR = os.path.join(CHECKPOINT_DIR, 'weights')
os.makedirs(os.path.join(WEIGHTS_DIR, 'af'), exist_ok=True)
os.makedirs(os.path.join(WEIGHTS_DIR, 'bs'), exist_ok=True)

import shutil
for name in os.listdir(CHECKPOINT_DIR):
    srcp = os.path.join(CHECKPOINT_DIR, name)
    if not os.path.isfile(srcp):
        continue
    if name.startswith('af_'):
        shutil.copy2(srcp, os.path.join(WEIGHTS_DIR, 'af', name))
    elif name.startswith('bs_'):
        shutil.copy2(srcp, os.path.join(WEIGHTS_DIR, 'bs', name))
print('веса разложены в', WEIGHTS_DIR)
for sub in ('af', 'bs'):
    print(' ', sub + '/:', sorted(os.listdir(os.path.join(WEIGHTS_DIR, sub))))

In [ ]:
# освобождаем всё обучающее, чтобы замер шёл в тех же условиях, что у организаторов
for _n in ['bs_lgb_model', 'refiner_models', 'af_models', 'X_bs_va', 'y_bs_va',
            'X_af_tr', 'y_af_tr', 'X_af_va', 'y_af_va', 'val_lgb_proba',
            'val_masks', 'val_valid']:
    globals().pop(_n, None)
import gc
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
print('память освобождена')

In [ ]:
INFERENCE_SECONDS = None

with stage('11. ИНФЕРЕНС НА ТЕСТЕ'):
    _t_inf = time.time()

    test_src = resolve_test_source(DATASET_DIR)
    sample_submission = test_src.read_csv('sample_submission.csv')

    af_art = load_af_artifacts(os.path.join(WEIGHTS_DIR, 'af'), device=DEVICE)
    bs_art = load_bs_artifacts(os.path.join(WEIGHTS_DIR, 'bs'), device=DEVICE)
    print('AF: моделей', len(af_art['models']), '| порог', af_art['global_threshold'])
    print('BS: стратегия', bs_art['strategy'], '| порог', bs_art['burn_threshold'],
          '| alpha', bs_art['refiner_alpha'], '| min_blob', bs_art['min_blob_size'])

    af_ids = sample_submission.loc[sample_submission['chip_id'].str.startswith('AF_'), 'chip_id'].unique()
    bs_ids = sample_submission.loc[sample_submission['chip_id'].str.startswith('BS_'), 'chip_id'].unique()

    rows = []
    _t0 = time.time()
    for chip_id in tqdm(af_ids, desc='AF инференс'):
        mask = predict_af_mask(chip_id, af_art['models'], af_art['cnn_model'],
                                af_art['global_threshold'], af_art['per_landcover_thresholds'],
                                test_src, device=DEVICE)
        rows.append(submission_row(chip_id, 1, mask))
    _af_time = time.time() - _t0

    _t0 = time.time()
    for chip_id in tqdm(bs_ids, desc='BS инференс'):
        mask = predict_bs_mask(
            chip_id, test_src,
            lgb_model=bs_art['lgb_model'], unet_models=bs_art['unet_models'],
            refiner_models=bs_art['refiner_models'], strategy=bs_art['strategy'],
            burn_threshold=bs_art['burn_threshold'], min_blob_size=bs_art['min_blob_size'],
            severity_weights=bs_art['severity_weights'], refiner_alpha=bs_art['refiner_alpha'],
            device=DEVICE, use_tta=BS_TTA)
        rows.extend(submission_row(chip_id, c, mask == c) for c in (1, 2, 3))
    _bs_time = time.time() - _t0

    submission = pd.DataFrame(rows)
    submission_path = os.path.join(CHECKPOINT_DIR, 'submission.csv')
    submission.to_csv(submission_path, index=False)
    test_src.close()
    INFERENCE_SECONDS = time.time() - _t_inf

print(f'AF: {_af_time:.1f} с на {len(af_ids)} чипов | BS: {_bs_time:.1f} с на {len(bs_ids)} чипов')
print('submission:', submission_path, submission.shape)
problems = validate_submission(submission, sample_submission)
print('проблемы валидации (пусто = ок):', problems)
assert not problems, problems

## 10. Отчёт по времени и что заливать

In [ ]:
print('=' * 62)
print('ВРЕМЯ ПО ЭТАПАМ')
print('=' * 62)
for k, v in TIMINGS.items():
    print(f'  {k:<42s} {v/60:6.1f} мин')
print(f'  {"ВСЕГО":<42s} {sum(TIMINGS.values())/60:6.1f} мин')
print()
print('=' * 62)
print('СКОРОСТЬ ИНФЕРЕНСА (за это начисляют баллы)')
print('=' * 62)
print(f'  полный инференс (веса с диска + весь тест + запись csv): {INFERENCE_SECONDS:.1f} с')
_brackets = [(30, 8), (300, 6), (720, 4), (1800, 2), (3600, 1)]
_pts = next((p for lim, p in _brackets if INFERENCE_SECONDS <= lim), 0)
print(f'  бракет: {_pts} баллов из 8')
print()
print('  ВАЖНО: это время на ЭТОЙ машине. У организаторов H100 и, скорее всего,\n'
      '  более быстрый CPU. Часть пайплайна (признаки + LightGBM) считается на CPU\n'
      '  и от GPU не ускоряется, поэтому замер здесь - консервативная оценка.')

In [ ]:
print('ЧТО ЗАЛИВАТЬ:')
print()
print('1. На платформу как сабмишн:')
print('  ', submission_path)
print()
print('2. В репозиторий, в solution/weights/ (скачать из Google Drive):')
for sub in ('af', 'bs'):
    d = os.path.join(WEIGHTS_DIR, sub)
    total = sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d))
    print(f'   weights/{sub}/  ({total/1e6:.1f} МБ):')
    for f in sorted(os.listdir(d)):
        print(f'     - {f}  ({os.path.getsize(os.path.join(d, f))/1e6:.2f} МБ)')
print()
print('3. Проверить у себя, что standalone-скрипт даёт тот же файл:')
print('   python inference.py --data-dir <путь к test> --weights-dir weights '
      '--output submission.csv')
src.close()